In [3]:
import torch
import torch.utils.data as data
from torchvision import transforms
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
import numpy as np
from PIL import Image
import os
import json
import random
from pathlib import Path
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from torchvision.transforms import functional as F
from sklearn.model_selection import train_test_split
from tqdm import tqdm
import shutil
import xml.etree.ElementTree as ET
from typing import Dict, List, Tuple, Optional
from collections import defaultdict


# ====================================================================
# SMART PATH FINDER - Same logic as YOLO code
# ====================================================================
def find_verification_files(project_root: Path, iteration: str) -> Tuple[Optional[Path], Optional[Path]]:
    """
    Intelligently search for verification result files in multiple possible locations.
    
    Args:
        project_root: Project root directory
        iteration: 'it1', 'it2', or 'it3'
    
    Returns:
        (true_positive_json_path, false_positive_json_path) or (None, None)
    """
    
    if iteration == 'it1':
        search_paths = [
            project_root / "wsi_global_xmls_test_batch/verified_comparison_results1/reports",
            project_root / "wsi_global_xmls_test_batch1/verified_comparison_results/reports",
            project_root / "wsi_global_xmls_test_batch1/verified_comparison_results1/reports",
        ]
    elif iteration == 'it2':
        search_paths = [
            project_root / "wsi_global_xmls_test_batch/verified_comparison_results2/reports",
            project_root / "wsi_global_xmls_test_batch2/verified_comparison_results/reports",
            project_root / "wsi_global_xmls_test_batch2/verified_comparison_results2/reports",
        ]
    elif iteration == 'it3':
        search_paths = [
            project_root / "wsi_global_xmls_test_batch/verified_comparison_results3/reports",
            project_root / "wsi_global_xmls_test_batch3_clean_check/verified_comparison_results/reports",
            project_root / "wsi_global_xmls_test_batch3_clean_check/verified_comparison_results3/reports",
            project_root / "wsi_global_xmls_test_batch3/verified_comparison_results/reports",
        ]
    else:
        return None, None
    
    for base_path in search_paths:
        tp_path = base_path / "true_positive_patches.json"
        fp_path = base_path / "false_positive_patches.json"
        
        if tp_path.exists() and fp_path.exists():
            return tp_path, fp_path
    
    return None, None


def find_xml_directory(project_root: Path, iteration: str) -> Optional[Path]:
    """Find the XML directory for a given iteration."""
    if iteration == 'it1':
        search_paths = [
            project_root / "wsi_global_xmls_test_batch/Verified_xml1_full",
            project_root / "wsi_global_xmls_test_batch1/Verified_xml_full",
        ]
    elif iteration == 'it2':
        search_paths = [
            project_root / "wsi_global_xmls_test_batch/Verified_xml2_full",
            project_root / "wsi_global_xmls_test_batch2/Verified_xml_full",
        ]
    elif iteration == 'it3':
        search_paths = [
            project_root / "wsi_global_xmls_test_batch/Verified_xml3_full",
            project_root / "wsi_global_xmls_test_batch3_clean_check/Verified_xml3_full",
            project_root / "wsi_global_xmls_test_batch3_clean_check/Verified_xml_full",
            project_root / "wsi_global_xmls_test_batch3/Verified_xml_full",
        ]
    else:
        return None
    
    for path in search_paths:
        if path.exists():
            return path
    
    return None


# ====================================================================
# CONFIGURATION WITH SMART PATH DISCOVERY
# ====================================================================
class Config:
    def __init__(self):
        # Base paths
        self.PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
        self.BASE_DIR = self.PROJECT_ROOT / "master-data/separated_patches"
        self.OUTPUT_DIR = self.PROJECT_ROOT / "rcnn_optimal_final"  # NEW OUTPUT DIRECTORY
        
        # Ground Truth Data
        self.ORIGINAL_POSITIVE_IMAGES = self.BASE_DIR / "positive_new/images"
        self.ORIGINAL_POSITIVE_LABELS = self.BASE_DIR / "positive_new/labels"
        self.ORIGINAL_NEGATIVE_IMAGES = self.BASE_DIR / "negative/images"
        
        # Image sources (same for all iterations as per YOLO code)
        self.IT1_IMG_SOURCE = self.BASE_DIR / "test_data_full/images"
        self.IT2_IMG_SOURCE = self.BASE_DIR / "test_data_full/images"
        self.IT3_IMG_SOURCE = self.BASE_DIR / "test_data_full/images"
        
        # Training parameters
        self.IMG_SIZE = 512
        self.BATCH_SIZE = 8
        self.NUM_EPOCHS = 50
        self.INITIAL_LR = 0.003
        self.WEIGHT_DECAY = 0.0005
        self.NUM_CLASSES = 2
        self.CONF_THRESHOLD = 0.3
        
        # Data split
        self.TRAIN_RATIO = 0.90
        self.VAL_RATIO = 0.10
        self.RANDOM_SEED = 42
        
        # Negative sampling - REDUCED to avoid disk quota issues
        # Use symlinks instead of copying files
        self.HARD_NEG_RATIO_IT1 = 0.5  # Reduced from 1.0
        self.HARD_NEG_RATIO_IT2 = 0.5  # Reduced from 1.0
        self.HARD_NEG_RATIO_IT3 = 0.5  # Reduced from 1.0
        self.EASY_NEG_COUNT = 300      # Reduced from 500
        
        # Use symlinks to save disk space
        self.USE_SYMLINKS = True
        
        # Warmup parameters
        self.WARMUP_EPOCHS = 5
        self.WARMUP_FACTOR = 0.1
        
        # Min box size
        self.MIN_BOX_SIZE = 2
        
        # Auto-discover paths
        self._discover_paths()
    
    def _discover_paths(self):
        """Auto-discover verification file locations"""
        print("\n🔍 AUTO-DISCOVERING VERIFICATION FILE LOCATIONS...")
        
        # IT1
        tp1, fp1 = find_verification_files(self.PROJECT_ROOT, 'it1')
        self.VERIF_IT1_TP = tp1
        self.VERIF_IT1_FP = fp1
        self.XML_SOURCE_IT1 = find_xml_directory(self.PROJECT_ROOT, 'it1')
        
        # IT2
        tp2, fp2 = find_verification_files(self.PROJECT_ROOT, 'it2')
        self.VERIF_IT2_TP = tp2
        self.VERIF_IT2_FP = fp2
        self.XML_SOURCE_IT2 = find_xml_directory(self.PROJECT_ROOT, 'it2')
        
        # IT3
        tp3, fp3 = find_verification_files(self.PROJECT_ROOT, 'it3')
        self.VERIF_IT3_TP = tp3
        self.VERIF_IT3_FP = fp3
        self.XML_SOURCE_IT3 = find_xml_directory(self.PROJECT_ROOT, 'it3')
        
        self._report_discovered_paths()
    
    def _report_discovered_paths(self):
        """Report which files were found"""
        print("\n📁 DISCOVERED FILE LOCATIONS:")
        
        for it_name, tp, fp, xml_dir in [
            ("IT1", self.VERIF_IT1_TP, self.VERIF_IT1_FP, self.XML_SOURCE_IT1),
            ("IT2", self.VERIF_IT2_TP, self.VERIF_IT2_FP, self.XML_SOURCE_IT2),
            ("IT3", self.VERIF_IT3_TP, self.VERIF_IT3_FP, self.XML_SOURCE_IT3),
        ]:
            print(f"\n{it_name}:")
            
            if tp:
                print(f"   ✅ True Positives:  {tp.parent.parent.name}/reports/")
            else:
                print(f"   ❌ True Positives:  NOT FOUND")
            
            if fp:
                print(f"   ✅ False Positives: {fp.parent.parent.name}/reports/")
            else:
                print(f"   ❌ False Positives: NOT FOUND")
            
            if xml_dir:
                print(f"   ✅ XML Directory:   {xml_dir.name}/")
            else:
                print(f"   ❌ XML Directory:   NOT FOUND")
        print()


# ====================================================================
# STATISTICS TRACKER
# ====================================================================
class StatsTracker:
    def __init__(self):
        self.stats = defaultdict(int)
        
    def increment(self, key: str, count: int = 1):
        self.stats[key] += count
        
    def report(self):
        print("\n" + "="*80)
        print("📊 DATASET STATISTICS REPORT")
        print("="*80)
        print("\n🟢 POSITIVE SAMPLES (Bacteria Present):")
        print(f"   Ground Truth (Original):     {self.stats['pos_original']:5d}")
        print(f"   Verified IT1 Recovered:      {self.stats['pos_it1']:5d}")
        print(f"   Verified IT2 Recovered:      {self.stats['pos_it2']:5d}")
        print(f"   Verified IT3 Recovered:      {self.stats['pos_it3']:5d}")
        print(f"   {'─'*50}")
        print(f"   Total Positives:             {self.stats['pos_total']:5d}")
        
        print("\n🔴 NEGATIVE SAMPLES (Empty/Artifacts):")
        print(f"   Hard Negatives IT1:          {self.stats['neg_hard_it1']:5d}")
        print(f"   Hard Negatives IT2:          {self.stats['neg_hard_it2']:5d}")
        print(f"   Hard Negatives IT3:          {self.stats['neg_hard_it3']:5d}")
        print(f"   Easy Negatives (Clean Bkg):  {self.stats['neg_easy']:5d}")
        print(f"   {'─'*50}")
        total_hard = (self.stats['neg_hard_it1'] + self.stats['neg_hard_it2'] + 
                     self.stats['neg_hard_it3'])
        print(f"   Total Hard Negatives:        {total_hard:5d}")
        print(f"   Total Negatives:             {self.stats['neg_total']:5d}")
        
        if self.stats['pos_total'] > 0:
            ratio = self.stats['neg_total'] / self.stats['pos_total']
            print(f"\n⚖️  Negative:Positive Ratio:     {ratio:.3f}:1")
        
        total = self.stats['pos_total'] + self.stats['neg_total']
        if total > 0:
            pos_pct = 100 * self.stats['pos_total'] / total
            print(f"📈 Positive Percentage:          {pos_pct:.1f}%")
        
        # Bacteria instance counts
        print(f"\n🦠 BACTERIA INSTANCES:")
        print(f"   Train: {self.stats['bacteria_train']:5d}")
        print(f"   Val:   {self.stats['bacteria_val']:5d}")
        print(f"   Total: {self.stats['bacteria_train'] + self.stats['bacteria_val']:5d}")
        
        print("="*80 + "\n")


# ====================================================================
# DATA MANAGER WITH XML PARSING
# ====================================================================
class DataManager:
    def __init__(self, config: Config):
        self.cfg = config
        self.used_patch_names = set()
        self.xml_cache = {}
        self.stats = StatsTracker()
    
    def load_json(self, path: Optional[Path]) -> List[Dict]:
        """Load JSON with None-safe handling"""
        if path is None or not path.exists():
            return []
        with open(path, 'r') as f:
            return json.load(f)
    
    def parse_xml_for_wsi(self, wsi_id: str, xml_dir: Optional[Path], suffix: str) -> List[Dict]:
        """Parse XML annotations for a WSI"""
        if xml_dir is None:
            return []
        
        cache_key = f"{wsi_id}_{suffix}"
        if cache_key in self.xml_cache:
            return self.xml_cache[cache_key]
        
        candidates = [xml_dir / f"{wsi_id}{suffix}", xml_dir / f"{wsi_id}.xml"]
        root = None
        for xml_path in candidates:
            if xml_path.exists():
                try:
                    tree = ET.parse(xml_path)
                    root = tree.getroot()
                    break
                except ET.ParseError:
                    pass
        
        annotations = []
        if root:
            for region in root.findall('.//Region'):
                vertices = [(int(v.get('X')), int(v.get('Y'))) for v in region.findall('.//Vertex')]
                if len(vertices) >= 2:
                    xs, ys = zip(*vertices)
                    annotations.append({
                        'x_min': min(xs), 'y_min': min(ys),
                        'x_max': max(xs), 'y_max': max(ys)
                    })
        
        self.xml_cache[cache_key] = annotations
        return annotations
    
    def get_labels_for_patch(self, wsi_id: str, offset_x: int, offset_y: int,
                            xml_dir: Optional[Path], xml_suffix: str) -> List[str]:
        """Generate YOLO format labels for a patch from XML annotations"""
        annotations = self.parse_xml_for_wsi(wsi_id, xml_dir, xml_suffix)
        if not annotations:
            return []
        
        yolo_labels = []
        patch_x1, patch_y1 = offset_x, offset_y
        patch_x2, patch_y2 = offset_x + self.cfg.IMG_SIZE, offset_y + self.cfg.IMG_SIZE
        
        for ann in annotations:
            if (ann['x_max'] > patch_x1 and ann['x_min'] < patch_x2 and
                ann['y_max'] > patch_y1 and ann['y_min'] < patch_y2):
                
                box_x_min = max(0, ann['x_min'] - patch_x1)
                box_y_min = max(0, ann['y_min'] - patch_y1)
                box_x_max = min(self.cfg.IMG_SIZE, ann['x_max'] - patch_x1)
                box_y_max = min(self.cfg.IMG_SIZE, ann['y_max'] - patch_y1)
                
                w = box_x_max - box_x_min
                h = box_y_max - box_y_min
                
                if w >= self.cfg.MIN_BOX_SIZE and h >= self.cfg.MIN_BOX_SIZE:
                    cx = (box_x_min + w / 2) / self.cfg.IMG_SIZE
                    cy = (box_y_min + h / 2) / self.cfg.IMG_SIZE
                    nw = w / self.cfg.IMG_SIZE
                    nh = h / self.cfg.IMG_SIZE
                    yolo_labels.append(f"0 {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}")
        
        return yolo_labels
    
    def collect_positives(self) -> List[Dict]:
        """Collect all positive samples from ground truth + IT1 + IT2 + IT3"""
        samples = []
        print("\n🔍 Collecting Positive Samples...")
        
        # 1. Ground Truth
        print("   📦 Ground Truth (Original)...")
        for img in self.cfg.ORIGINAL_POSITIVE_IMAGES.glob("*.png"):
            lbl = self.cfg.ORIGINAL_POSITIVE_LABELS / f"{img.stem}.txt"
            if lbl.exists():
                samples.append({
                    'img': img,
                    'lbl_type': 'file',
                    'src': lbl
                })
                self.used_patch_names.add(img.name)
                self.stats.increment('pos_original')
        
        # 2. Verified Positives from IT1, IT2, IT3
        json_sources = [
            (self.cfg.VERIF_IT1_TP, self.cfg.IT1_IMG_SOURCE, self.cfg.XML_SOURCE_IT1, '_PO.xml', 'pos_it1', 'IT1'),
            (self.cfg.VERIF_IT2_TP, self.cfg.IT2_IMG_SOURCE, self.cfg.XML_SOURCE_IT2, '_PO2.xml', 'pos_it2', 'IT2'),
            (self.cfg.VERIF_IT3_TP, self.cfg.IT3_IMG_SOURCE, self.cfg.XML_SOURCE_IT3, '_PO3.xml', 'pos_it3', 'IT3'),
        ]
        
        for json_path, img_root, xml_dir, suffix, stat_key, it_name in json_sources:
            items = self.load_json(json_path)
            if not items:
                print(f"   ⚠️  {it_name}: No verification data found (skipping)")
                continue
            
            print(f"   📦 {it_name}: Processing {len(items)} verified patches...")
            added = 0
            
            for item in tqdm(items, desc=f"      {it_name}", leave=False):
                img_path = img_root / item['patch']
                if not img_path.exists() or img_path.name in self.used_patch_names:
                    continue
                
                labels = self.get_labels_for_patch(
                    item.get('wsi_id'),
                    item.get('offset_x'),
                    item.get('offset_y'),
                    xml_dir,
                    suffix
                )
                
                if labels:
                    samples.append({
                        'img': img_path,
                        'lbl_type': 'generated',
                        'labels': labels
                    })
                    self.used_patch_names.add(img_path.name)
                    self.stats.increment(stat_key)
                    added += 1
            
            print(f"      → Added {added} positive patches")
        
        self.stats.stats['pos_total'] = len(samples)
        print(f"\n   ✅ Total Positive Samples: {len(samples)}")
        return samples
    
    def collect_negatives(self) -> List[Dict]:
        """Collect negative samples: hard negatives from IT1+IT2+IT3 + some easy negatives"""
        samples = []
        print(f"\n🔍 Collecting Negative Samples...")
        
        # 1. HARD NEGATIVES from each iteration
        hard_sources = [
            (self.cfg.VERIF_IT1_FP, self.cfg.IT1_IMG_SOURCE, self.cfg.HARD_NEG_RATIO_IT1, 'neg_hard_it1', 'IT1'),
            (self.cfg.VERIF_IT2_FP, self.cfg.IT2_IMG_SOURCE, self.cfg.HARD_NEG_RATIO_IT2, 'neg_hard_it2', 'IT2'),
            (self.cfg.VERIF_IT3_FP, self.cfg.IT3_IMG_SOURCE, self.cfg.HARD_NEG_RATIO_IT3, 'neg_hard_it3', 'IT3'),
        ]
        
        for json_path, img_root, ratio, stat_key, it_name in hard_sources:
            items = self.load_json(json_path)
            
            if not items:
                print(f"   ⚠️  {it_name} Hard Negatives: No data found (skipping)")
                continue
            
            # Sample based on ratio
            num_to_use = int(len(items) * ratio)
            items_to_use = random.sample(items, num_to_use) if num_to_use < len(items) else items
            
            print(f"   📦 {it_name} Hard Negatives: Using {len(items_to_use)}/{len(items)} patches...")
            added = 0
            
            for item in items_to_use:
                img_path = img_root / item['patch']
                if img_path.exists() and img_path.name not in self.used_patch_names:
                    samples.append({
                        'img': img_path,
                        'lbl_type': 'empty'
                    })
                    self.used_patch_names.add(img_path.name)
                    self.stats.increment(stat_key)
                    added += 1
            
            print(f"      → Added {added} hard negative patches")
        
        # 2. EASY NEGATIVES (original clean backgrounds)
        print(f"   📦 Easy Negatives: Sampling {self.cfg.EASY_NEG_COUNT} patches...")
        easy_negs = [x for x in self.cfg.ORIGINAL_NEGATIVE_IMAGES.glob("*.png")
                     if x.name not in self.used_patch_names]
        random.shuffle(easy_negs)
        
        for img in easy_negs[:self.cfg.EASY_NEG_COUNT]:
            samples.append({
                'img': img,
                'lbl_type': 'empty'
            })
            self.used_patch_names.add(img.name)
            self.stats.increment('neg_easy')
        
        print(f"      → Added {self.stats.stats['neg_easy']} easy negative patches")
        
        self.stats.stats['neg_total'] = len(samples)
        print(f"\n   ✅ Total Negative Samples: {len(samples)}")
        return samples
    
    def organize_dataset(self):
        """Organize the complete dataset for training"""
        print("\n" + "="*80)
        print("🗂️  ORGANIZING FASTER R-CNN DATASET")
        print("="*80)
        
        # Collect all data
        positives = self.collect_positives()
        negatives = self.collect_negatives()
        
        # Report statistics
        self.stats.report()
        
        # Combine and shuffle
        all_data = positives + negatives
        random.shuffle(all_data)
        
        # Train/Val split
        train_data, val_data = train_test_split(
            all_data,
            test_size=self.cfg.VAL_RATIO,
            random_state=self.cfg.RANDOM_SEED,
            stratify=[d['lbl_type'] != 'empty' for d in all_data]
        )
        
        print(f"📊 Train/Val Split:")
        print(f"   Train: {len(train_data):5d} patches")
        print(f"   Val:   {len(val_data):5d} patches")
        
        # Create output directories
        if self.cfg.OUTPUT_DIR.exists():
            print(f"\n⚠️  Output directory exists: {self.cfg.OUTPUT_DIR}")
            response = input("   Delete and recreate? (y/n): ").strip().lower()
            if response != 'y':
                print("   Aborting...")
                exit(1)
            shutil.rmtree(self.cfg.OUTPUT_DIR)
        
        dirs = {
            'train_images': self.cfg.OUTPUT_DIR / 'images' / 'train',
            'val_images': self.cfg.OUTPUT_DIR / 'images' / 'val',
            'train_labels': self.cfg.OUTPUT_DIR / 'labels' / 'train',
            'val_labels': self.cfg.OUTPUT_DIR / 'labels' / 'val',
        }
        
        for d in dirs.values():
            d.mkdir(parents=True, exist_ok=True)
        
        # Write data
        print(f"\n📝 Writing dataset files...")
        if self.cfg.USE_SYMLINKS:
            print("   💡 Using symbolic links to save disk space")
        self._write_split(train_data, dirs['train_images'], dirs['train_labels'], 'train')
        self._write_split(val_data, dirs['val_images'], dirs['val_labels'], 'val')
        
        print(f"\n✅ Dataset organized in: {self.cfg.OUTPUT_DIR}")
        return len(train_data), len(val_data)
    
    def _write_split(self, dataset, img_dir, lbl_dir, split_name):
        """Write images and labels for a data split - using symlinks to save space"""
        bacteria_count = 0
        
        for data in tqdm(dataset, desc=f"   {split_name.capitalize()}"):
            # Use symlink for images if enabled, otherwise copy
            img_dest = img_dir / data['img'].name
            
            if self.cfg.USE_SYMLINKS:
                try:
                    # Create relative symlink
                    img_dest.symlink_to(data['img'].resolve())
                except FileExistsError:
                    pass
            else:
                shutil.copy2(data['img'], img_dest)
            
            # Create label file (must be real file, not symlink)
            lbl_file = lbl_dir / f"{data['img'].stem}.txt"
            
            if data['lbl_type'] == 'file':
                # Copy label file
                shutil.copy2(data['src'], lbl_file)
                with open(data['src'], 'r') as f:
                    bacteria_count += len([l for l in f if l.strip()])
            
            elif data['lbl_type'] == 'generated':
                # Write generated labels
                with open(lbl_file, 'w') as f:
                    f.write("\n".join(data['labels']))
                bacteria_count += len(data['labels'])
            
            else:  # empty
                # Create empty label file
                lbl_file.touch()
        
        # Update statistics
        if split_name == 'train':
            self.stats.increment('bacteria_train', bacteria_count)
        else:
            self.stats.increment('bacteria_val', bacteria_count)


# ====================================================================
# ENHANCED DATASET CLASS
# ====================================================================
class HPyloriDataset(data.Dataset):
    """Enhanced dataset with better augmentation and box validation"""
    
    def __init__(self, images_dir, labels_dir, transforms=None, augment=False):
        self.images_dir = images_dir
        self.labels_dir = labels_dir
        self.custom_transforms = transforms
        self.augment = augment
        self.images = sorted([f for f in os.listdir(images_dir)
                            if f.endswith(('.png', '.jpg', '.jpeg', '.PNG', '.JPG', '.JPEG'))])
        
        self.positive_count = 0
        self.negative_count = 0
        for img_name in self.images:
            label_name = os.path.splitext(img_name)[0] + ".txt"
            label_path = os.path.join(self.labels_dir, label_name)
            if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
                self.positive_count += 1
            else:
                self.negative_count += 1
        
        print(f"   Dataset loaded: {len(self.images)} images")
        print(f"      Positives: {self.positive_count}")
        print(f"      Negatives: {self.negative_count}")
    
    def __len__(self):
        return len(self.images)
    
    def __getitem__(self, idx):
        img_name = self.images[idx]
        img_path = os.path.join(self.images_dir, img_name)
        
        # Handle symlinks
        if os.path.islink(img_path):
            img_path = os.path.realpath(img_path)
        
        img = Image.open(img_path).convert("RGB")
        width, height = img.size
        
        label_name = os.path.splitext(img_name)[0] + ".txt"
        label_path = os.path.join(self.labels_dir, label_name)
        
        boxes = []
        labels = []
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                lines = f.readlines()
                for line in lines:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        cls, x_c, y_c, w, h = map(float, parts[:5])
                        
                        xmin = (x_c - w/2) * width
                        ymin = (y_c - h/2) * height
                        xmax = (x_c + w/2) * width
                        ymax = (y_c + h/2) * height
                        
                        xmin = max(0, min(xmin, width))
                        ymin = max(0, min(ymin, height))
                        xmax = max(0, min(xmax, width))
                        ymax = max(0, min(ymax, height))
                        
                        if xmax > xmin + 2 and ymax > ymin + 2:
                            boxes.append([xmin, ymin, xmax, ymax])
                            labels.append(1)
        
        # Enhanced augmentation for training
        if self.augment and len(boxes) > 0:
            img, boxes = self.apply_augmentation(img, boxes, width, height)
        
        if len(boxes) > 0:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)
            labels = torch.as_tensor(labels, dtype=torch.int64)
            area = (boxes[:, 3] - boxes[:, 1]) * (boxes[:, 2] - boxes[:, 0])
        else:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)
            area = torch.zeros((0,), dtype=torch.float32)
        
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": area,
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }
        
        if self.custom_transforms:
            img = self.custom_transforms(img)
        else:
            img = transforms.ToTensor()(img)
        
        return img, target
    
    def apply_augmentation(self, img, boxes, width, height):
        """Enhanced augmentation preserving small objects"""
        # Horizontal flip (50%)
        if random.random() > 0.5:
            img = F.hflip(img)
            boxes = [[width - box[2], box[1], width - box[0], box[3]] for box in boxes]
        
        # Vertical flip (50%)
        if random.random() > 0.5:
            img = F.vflip(img)
            boxes = [[box[0], height - box[3], box[2], height - box[1]] for box in boxes]
        
        # Small rotation (30% chance)
        if random.random() > 0.7:
            angle = random.uniform(-10, 10)
            img = F.rotate(img, angle)
        
        # Color jitter (70% chance)
        if random.random() > 0.3:
            brightness = random.uniform(0.85, 1.15)
            contrast = random.uniform(0.85, 1.15)
            saturation = random.uniform(0.85, 1.15)
            img = F.adjust_brightness(img, brightness)
            img = F.adjust_contrast(img, contrast)
            img = F.adjust_saturation(img, saturation)
        
        # Gaussian blur (20% chance)
        if random.random() > 0.8:
            img = F.gaussian_blur(img, kernel_size=3)
        
        return img, boxes


# ====================================================================
# OPTIMIZED MODEL FOR SMALL OBJECT DETECTION
# ====================================================================
def get_optimized_model(num_classes=2):
    """Heavily optimized Faster R-CNN for tiny bacteria (10-50 pixels)"""
    anchor_generator = AnchorGenerator(
        sizes=((4,), (8,), (16,), (32,), (64,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5
    )
    
    model = fasterrcnn_resnet50_fpn(
        weights='DEFAULT',
        trainable_backbone_layers=4,
        min_size=800,
        max_size=1333
    )
    
    model.rpn.anchor_generator = anchor_generator
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    model.rpn.pre_nms_top_n_train = 3000
    model.rpn.pre_nms_top_n_test = 1500
    model.rpn.post_nms_top_n_train = 3000
    model.rpn.post_nms_top_n_test = 1500
    model.rpn.nms_thresh = 0.6
    model.rpn.score_thresh = 0.0
    model.rpn.fg_iou_thresh = 0.5
    model.rpn.bg_iou_thresh = 0.3
    
    model.roi_heads.nms_thresh = 0.2
    model.roi_heads.score_thresh = 0.01
    model.roi_heads.detections_per_img = 200
    model.roi_heads.fg_iou_thresh = 0.4
    model.roi_heads.bg_iou_thresh = 0.3
    
    return model


def collate_fn(batch):
    """Custom collate function"""
    return tuple(zip(*batch))


# ====================================================================
# TRAINING FUNCTIONS
# ====================================================================
def train_one_epoch(model, optimizer, data_loader, device, epoch, warmup_scheduler=None):
    """Enhanced training with gradient accumulation and loss monitoring"""
    model.train()
    total_loss = 0
    loss_classifier_sum = 0
    loss_box_reg_sum = 0
    loss_objectness_sum = 0
    loss_rpn_box_reg_sum = 0
    valid_batches = 0
    
    pbar = tqdm(enumerate(data_loader), total=len(data_loader),
                desc=f"Epoch {epoch}",
                bar_format='{desc}: {percentage:3.0f}%|{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}]')
    
    for i, (images, targets) in pbar:
        try:
            images = list(image.to(device) for image in images)
            targets = [{k: v.to(device) for k, v in t.items()} for t in targets]
            
            loss_dict = model(images, targets)
            losses = sum(loss for loss in loss_dict.values())
            
            if torch.isnan(losses) or torch.isinf(losses):
                print(f"  ⚠️ Skipping batch {i}: NaN/Inf loss")
                continue
            
            optimizer.zero_grad()
            losses.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=10.0)
            optimizer.step()
            
            if warmup_scheduler is not None:
                warmup_scheduler.step()
            
            total_loss += losses.item()
            loss_classifier_sum += loss_dict.get('loss_classifier', torch.tensor(0)).item()
            loss_box_reg_sum += loss_dict.get('loss_box_reg', torch.tensor(0)).item()
            loss_objectness_sum += loss_dict.get('loss_objectness', torch.tensor(0)).item()
            loss_rpn_box_reg_sum += loss_dict.get('loss_rpn_box_reg', torch.tensor(0)).item()
            valid_batches += 1
            
            current_lr = optimizer.param_groups[0]['lr']
            pbar.set_postfix({'loss': f'{losses.item():.4f}', 'lr': f'{current_lr:.6f}'})
            
        except RuntimeError as e:
            print(f"  ⚠️ Skipping batch {i}: {str(e)[:100]}")
            continue
    
    if valid_batches == 0:
        return float('inf'), {}
    
    avg_loss = total_loss / valid_batches
    loss_breakdown = {
        'classifier': loss_classifier_sum / valid_batches,
        'box_reg': loss_box_reg_sum / valid_batches,
        'objectness': loss_objectness_sum / valid_batches,
        'rpn_box': loss_rpn_box_reg_sum / valid_batches
    }
    
    print(f"\nEpoch {epoch} Summary:")
    print(f"  Total Loss:    {avg_loss:.4f}")
    print(f"  Classifier:    {loss_breakdown['classifier']:.4f}")
    print(f"  Box Reg:       {loss_breakdown['box_reg']:.4f}")
    print(f"  Objectness:    {loss_breakdown['objectness']:.4f}")
    print(f"  RPN Box:       {loss_breakdown['rpn_box']:.4f}")
    
    return avg_loss, loss_breakdown


@torch.no_grad()
def evaluate(model, data_loader, device, conf_threshold=0.3, iou_threshold=0.3):
    """Enhanced evaluation"""
    model.eval()
    
    all_predictions = []
    all_targets = []
    
    print(f"\n{'='*80}")
    print("EVALUATING MODEL")
    print(f"{'='*80}")
    
    for images, targets in tqdm(data_loader, desc="Evaluating"):
        images = list(img.to(device) for img in images)
        outputs = model(images)
        
        for output, target in zip(outputs, targets):
            all_predictions.append({
                'boxes': output['boxes'].cpu().numpy(),
                'scores': output['scores'].cpu().numpy(),
                'labels': output['labels'].cpu().numpy()
            })
            all_targets.append({
                'boxes': target['boxes'].cpu().numpy(),
                'labels': target['labels'].cpu().numpy()
            })
    
    print(f"\n{'EVALUATION RESULTS':-^80}")
    
    for iou_thresh in [0.3, 0.5, 0.75]:
        metrics = calculate_detection_metrics(
            all_predictions, all_targets,
            conf_threshold=conf_threshold,
            iou_threshold=iou_thresh
        )
        
        print(f"\n{'─'*80}")
        print(f"IoU Threshold: {iou_thresh}")
        print(f"{'─'*80}")
        print(f"Precision: {metrics['precision']:.4f} | Recall: {metrics['recall']:.4f} | F1: {metrics['f1']:.4f}")
        print(f"TP: {metrics['tp']} | FP: {metrics['fp']} | FN: {metrics['fn']}")
    
    final_metrics = calculate_detection_metrics(
        all_predictions, all_targets,
        conf_threshold=conf_threshold,
        iou_threshold=0.3
    )
    
    print(f"\n{'PRIMARY METRICS (IoU=0.3)':-^80}")
    print(f"Precision: {final_metrics['precision']:.4f}")
    print(f"Recall:    {final_metrics['recall']:.4f}")
    print(f"F1-Score:  {final_metrics['f1']:.4f}")
    print(f"{'='*80}\n")
    
    return final_metrics


def calculate_detection_metrics(predictions, targets, conf_threshold=0.3, iou_threshold=0.3):
    """Calculate comprehensive detection metrics"""
    tp = 0
    fp = 0
    fn = 0
    
    for pred, target in zip(predictions, targets):
        pred_boxes = pred['boxes']
        pred_scores = pred['scores']
        gt_boxes = target['boxes']
        
        mask = pred_scores >= conf_threshold
        pred_boxes_filtered = pred_boxes[mask]
        
        if len(gt_boxes) == 0:
            fp += len(pred_boxes_filtered)
        elif len(pred_boxes_filtered) == 0:
            fn += len(gt_boxes)
        else:
            matched_gt = set()
            
            for pred_box in pred_boxes_filtered:
                best_iou = 0
                best_gt_idx = -1
                
                for gt_idx, gt_box in enumerate(gt_boxes):
                    if gt_idx in matched_gt:
                        continue
                    
                    iou = calculate_iou(pred_box, gt_box)
                    if iou > best_iou:
                        best_iou = iou
                        best_gt_idx = gt_idx
                
                if best_iou >= iou_threshold:
                    tp += 1
                    matched_gt.add(best_gt_idx)
                else:
                    fp += 1
            
            fn += len(gt_boxes) - len(matched_gt)
    
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'precision': precision,
        'recall': recall,
        'f1': f1
    }


def calculate_iou(box1, box2):
    """Calculate IoU between two boxes"""
    x1_min, y1_min, x1_max, y1_max = box1
    x2_min, y2_min, x2_max, y2_max = box2
    
    inter_x_min = max(x1_min, x2_min)
    inter_y_min = max(y1_min, y2_min)
    inter_x_max = min(x1_max, x2_max)
    inter_y_max = min(y1_max, y2_max)
    
    inter_width = max(0, inter_x_max - inter_x_min)
    inter_height = max(0, inter_y_max - inter_y_min)
    inter_area = inter_width * inter_height
    
    box1_area = (x1_max - x1_min) * (y1_max - y1_min)
    box2_area = (x2_max - x2_min) * (y2_max - y2_min)
    union_area = box1_area + box2_area - inter_area
    
    return inter_area / union_area if union_area > 0 else 0.0


def visualize_predictions(model, dataset, device, num_images=4, conf_threshold=0.3, save_path=None):
    """Visualize predictions"""
    model.eval()
    
    indices = np.random.choice(len(dataset), min(num_images, len(dataset)), replace=False)
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 16))
    axes = axes.flatten()
    
    for plot_idx, data_idx in enumerate(indices):
        if plot_idx >= len(axes):
            break
        
        img, target = dataset[data_idx]
        img_tensor = img.unsqueeze(0).to(device)
        
        with torch.no_grad():
            prediction = model(img_tensor)[0]
        
        img_np = img.permute(1, 2, 0).cpu().numpy()
        ax = axes[plot_idx]
        ax.imshow(img_np)
        
        gt_boxes = target['boxes'].cpu().numpy()
        for box in gt_boxes:
            x1, y1, x2, y2 = box
            rect = patches.Rectangle(
                (x1, y1), x2-x1, y2-y1,
                linewidth=2, edgecolor='lime', facecolor='none',
                linestyle='--', label='GT'
            )
            ax.add_patch(rect)
        
        boxes = prediction['boxes'].cpu().numpy()
        scores = prediction['scores'].cpu().numpy()
        
        detected = 0
        for box, score in zip(boxes, scores):
            if score > conf_threshold:
                detected += 1
                x1, y1, x2, y2 = box
                rect = patches.Rectangle(
                    (x1, y1), x2-x1, y2-y1,
                    linewidth=2, edgecolor='red', facecolor='none'
                )
                ax.add_patch(rect)
                ax.text(x1, y1-5, f'{score:.2f}',
                       color='red', fontsize=9, weight='bold',
                       bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.9))
        
        ax.axis('off')
        title = f'GT: {len(gt_boxes)} | Pred: {detected}'
        ax.set_title(title, fontsize=12, weight='bold')
    
    plt.tight_layout()
    
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
        print(f"  Saved: {save_path}")
    
    plt.close()


# ====================================================================
# MAIN TRAINING SCRIPT
# ====================================================================
def main():
    # Set random seeds
    random.seed(42)
    np.random.seed(42)
    torch.manual_seed(42)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(42)
    
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    
    print("\n" + "="*80)
    print("🔬 FASTER R-CNN FOR H. PYLORI DETECTION - ITERATION 3 FINAL")
    print("="*80)
    print(f"\nDevice: {device}")
    if torch.cuda.is_available():
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    
    # Initialize configuration
    config = Config()
    
    print("\n💡 DISK SPACE OPTIMIZATION:")
    print(f"   • Using symbolic links: {config.USE_SYMLINKS}")
    print(f"   • Hard negative ratio per iteration: {config.HARD_NEG_RATIO_IT1*100:.0f}%")
    print(f"   • Easy negatives: {config.EASY_NEG_COUNT}")
    
    # Organize dataset
    data_manager = DataManager(config)
    train_size, val_size = data_manager.organize_dataset()
    
    # Setup paths
    train_images_dir = config.OUTPUT_DIR / 'images' / 'train'
    train_labels_dir = config.OUTPUT_DIR / 'labels' / 'train'
    val_images_dir = config.OUTPUT_DIR / 'images' / 'val'
    val_labels_dir = config.OUTPUT_DIR / 'labels' / 'val'
    
    # Create datasets
    print("\n📂 Loading datasets...")
    train_dataset = HPyloriDataset(
        str(train_images_dir),
        str(train_labels_dir),
        augment=True
    )
    val_dataset = HPyloriDataset(
        str(val_images_dir),
        str(val_labels_dir),
        augment=False
    )
    
    # Data loaders
    train_loader = DataLoader(
        train_dataset, batch_size=config.BATCH_SIZE,
        shuffle=True, collate_fn=collate_fn,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    val_loader = DataLoader(
        val_dataset, batch_size=config.BATCH_SIZE,
        shuffle=False, collate_fn=collate_fn,
        num_workers=4, pin_memory=True, persistent_workers=True
    )
    
    # Create model
    print("\n🏗️  Building OPTIMIZED model...")
    model = get_optimized_model(config.NUM_CLASSES)
    model.to(device)
    print("✓ Model ready")
    print("  • Anchor sizes: 4, 8, 16, 32, 64 pixels")
    print("  • Max detections per image: 200")
    print("  • NMS threshold: 0.2")
    
    # Optimizer
    params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.SGD(
        params,
        lr=config.INITIAL_LR,
        momentum=0.9,
        weight_decay=config.WEIGHT_DECAY
    )
    
    # Warmup scheduler
    warmup_iters = len(train_loader) * config.WARMUP_EPOCHS
    warmup_scheduler = torch.optim.lr_scheduler.LinearLR(
        optimizer,
        start_factor=config.WARMUP_FACTOR,
        total_iters=warmup_iters
    )
    
    # Main scheduler
    main_scheduler = torch.optim.lr_scheduler.MultiStepLR(
        optimizer,
        milestones=[30, 50, 70, 85],
        gamma=0.5
    )
    
    print(f"\n📊 Training Configuration:")
    print(f"  Batch size: {config.BATCH_SIZE}")
    print(f"  Initial LR: {config.INITIAL_LR}")
    print(f"  Warmup epochs: {config.WARMUP_EPOCHS}")
    print(f"  Max epochs: {config.NUM_EPOCHS}")
    
    # Setup checkpoint directory
    models_dir = config.OUTPUT_DIR / 'models'
    models_dir.mkdir(exist_ok=True)
    checkpoint_path = models_dir / 'checkpoint_last.pth'
    
    # Check for checkpoint
    start_epoch = 1
    best_f1 = 0.0
    best_loss = float('inf')
    patience_counter = 0
    
    if checkpoint_path.exists():
        print(f"\n📂 Found checkpoint: {checkpoint_path}")
        response = input("Resume from checkpoint? (y/n): ").strip().lower()
        if response == 'y':
            print("Loading checkpoint...")
            checkpoint = torch.load(checkpoint_path)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            main_scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            start_epoch = checkpoint['epoch'] + 1
            best_f1 = checkpoint.get('best_f1', 0.0)
            best_loss = checkpoint.get('best_loss', float('inf'))
            patience_counter = checkpoint.get('patience_counter', 0)
            print(f"✓ Resumed from epoch {checkpoint['epoch']}")
    
    # Training loop
    print("\n" + "="*80)
    print("🚀 TRAINING START")
    print("="*80)
    
    patience = 25
    
    for epoch in range(start_epoch, config.NUM_EPOCHS + 1):
        print(f"\n{'='*80}")
        print(f"Epoch {epoch}/{config.NUM_EPOCHS}")
        print(f"{'='*80}")
        
        # Train
        train_loss, loss_breakdown = train_one_epoch(
            model, optimizer, train_loader, device, epoch,
            warmup_scheduler if epoch <= config.WARMUP_EPOCHS else None
        )
        
        # Step scheduler
        if epoch > config.WARMUP_EPOCHS:
            main_scheduler.step()
        
        # Validate every 3 epochs
        if epoch % 3 == 0 or epoch == config.NUM_EPOCHS:
            print(f"\n🔍 Validating...")
            metrics = evaluate(model, val_loader, device, config.CONF_THRESHOLD, iou_threshold=0.3)
            
            # Save best model
            current_f1 = metrics['f1']
            if current_f1 > best_f1:
                best_f1 = current_f1
                model_path = models_dir / 'rcnn_bacteria_best_f1.pth'
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'f1': best_f1,
                    'precision': metrics['precision'],
                    'recall': metrics['recall'],
                }, str(model_path))
                print(f"\n  ✅ NEW BEST F1! Saved: F1={best_f1:.4f}")
                patience_counter = 0
            else:
                patience_counter += 1
            
            # Visualize
            viz_path = config.OUTPUT_DIR / f'predictions_epoch_{epoch}.png'
            visualize_predictions(model, val_dataset, device, num_images=4,
                                conf_threshold=config.CONF_THRESHOLD, save_path=viz_path)
        
        # Save checkpoint
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': main_scheduler.state_dict(),
            'best_f1': best_f1,
            'best_loss': best_loss,
            'patience_counter': patience_counter,
        }, str(checkpoint_path))
        
        # Early stopping
        if patience_counter >= patience:
            print(f"\n⏹️  Early stopping at epoch {epoch}")
            break
    
    # Save final model
    final_model_path = models_dir / 'rcnn_bacteria_final.pth'
    torch.save(model.state_dict(), str(final_model_path))
    
    print("\n" + "="*80)
    print("✅ TRAINING COMPLETE!")
    print("="*80)
    print(f"\n📦 Models saved in: {models_dir}")
    print(f"  Best F1: {best_f1:.4f}")
    print(f"\n🎯 Use this model for testing:")
    print(f"  {models_dir / 'rcnn_bacteria_best_f1.pth'}")
    print("\n" + "="*80)


if __name__ == "__main__":
    main()


🔬 FASTER R-CNN FOR H. PYLORI DETECTION - ITERATION 3 FINAL

Device: cuda
GPU: Tesla V100-PCIE-32GB
GPU Memory: 31.73 GB

🔍 AUTO-DISCOVERING VERIFICATION FILE LOCATIONS...

📁 DISCOVERED FILE LOCATIONS:

IT1:
   ✅ True Positives:  verified_comparison_results1/reports/
   ✅ False Positives: verified_comparison_results1/reports/
   ✅ XML Directory:   Verified_xml1_full/

IT2:
   ✅ True Positives:  verified_comparison_results2/reports/
   ✅ False Positives: verified_comparison_results2/reports/
   ✅ XML Directory:   Verified_xml2_full/

IT3:
   ✅ True Positives:  verified_comparison_results3/reports/
   ✅ False Positives: verified_comparison_results3/reports/
   ✅ XML Directory:   Verified_xml3_full/


💡 DISK SPACE OPTIMIZATION:
   • Using symbolic links: True
   • Hard negative ratio per iteration: 50%
   • Easy negatives: 300

🗂️  ORGANIZING FASTER R-CNN DATASET

🔍 Collecting Positive Samples...
   📦 Ground Truth (Original)...
   📦 IT1: Processing 997 verified patches...


      → Added 997 positive patches
   📦 IT2: Processing 262 verified patches...


      → Added 28 positive patches
   📦 IT3: Processing 197 verified patches...


      → Added 180 positive patches

   ✅ Total Positive Samples: 2358

🔍 Collecting Negative Samples...
   📦 IT1 Hard Negatives: Using 1262/2525 patches...
      → Added 1234 hard negative patches
   📦 IT2 Hard Negatives: Using 149/299 patches...
      → Added 77 hard negative patches
   📦 IT3 Hard Negatives: Using 1340/2680 patches...
      → Added 1168 hard negative patches
   📦 Easy Negatives: Sampling 300 patches...
      → Added 300 easy negative patches

   ✅ Total Negative Samples: 2779

📊 DATASET STATISTICS REPORT

🟢 POSITIVE SAMPLES (Bacteria Present):
   Ground Truth (Original):      1153
   Verified IT1 Recovered:        997
   Verified IT2 Recovered:         28
   Verified IT3 Recovered:        180
   ──────────────────────────────────────────────────
   Total Positives:              2358

🔴 NEGATIVE SAMPLES (Empty/Artifacts):
   Hard Negatives IT1:           1234
   Hard Negatives IT2:             77
   Hard Negatives IT3:           1168
   Easy Negatives (Clean Bkg):    3

   Delete and recreate? (y/n):  y


...


Epoch 49 Summary:
  Total Loss:    0.0876
  Classifier:    0.0122
  Box Reg:       0.0147
  Objectness:    0.0052
  RPN Box:       0.0555

Epoch 50/50


Epoch 50: 100%|███████████████████████████████████████████| 578/578 [05:16<00:00]



Epoch 50 Summary:
  Total Loss:    0.0908
  Classifier:    0.0124
  Box Reg:       0.0149
  Objectness:    0.0055
  RPN Box:       0.0580

🔍 Validating...

EVALUATING MODEL


Evaluating: 100%|████████████████████████████████| 65/65 [00:14<00:00,  4.38it/s]



-------------------------------EVALUATION RESULTS-------------------------------

────────────────────────────────────────────────────────────────────────────────
IoU Threshold: 0.3
────────────────────────────────────────────────────────────────────────────────
Precision: 0.4887 | Recall: 0.5452 | F1: 0.5154
TP: 217 | FP: 227 | FN: 181

────────────────────────────────────────────────────────────────────────────────
IoU Threshold: 0.5
────────────────────────────────────────────────────────────────────────────────
Precision: 0.3761 | Recall: 0.4196 | F1: 0.3967
TP: 167 | FP: 277 | FN: 231

────────────────────────────────────────────────────────────────────────────────
IoU Threshold: 0.75
────────────────────────────────────────────────────────────────────────────────
Precision: 0.0991 | Recall: 0.1106 | F1: 0.1045
TP: 44 | FP: 400 | FN: 354

---------------------------PRIMARY METRICS (IoU=0.3)----------------------------
Precision: 0.4887
Recall:    0.5452
F1-Score:  0.5154

  Saved

In [ ]:
Test script

In [4]:
import cv2
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil
import json
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from collections import defaultdict
import xml.etree.ElementTree as ET
from xml.dom import minidom

# Import R-CNN model components
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision import transforms


# ====================================================================
# CONFIGURATION
# ====================================================================
class EvalConfig:
    PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
    
    # POINT TO THE R-CNN MODEL
    TRAIN_OUTPUT_DIR = PROJECT_ROOT / "rcnn_optimal_final"
    MODEL_PATH = TRAIN_OUTPUT_DIR / "models/rcnn_bacteria_best_f1.pth"
    
    # Output for this evaluation run
    EVAL_OUTPUT_DIR = TRAIN_OUTPUT_DIR / "evaluation_rcnn_candidates_check"
    
    # Data Sources (Unverified Data)
    # 1. Remaining Test Data (Slides we haven't fully checked)
    TEST_DATA_FULL = PROJECT_ROOT / "master-data/separated_patches/test_data_full/images"
    # 2. Original "Easy" Negatives (Check for missed bacteria)
    ORIGINAL_NEG_IMG = PROJECT_ROOT / "master-data/separated_patches/negative/images"
    
    # Inference Parameters (From R-CNN Training)
    BATCH_SIZE = 16  # R-CNN typically uses smaller batches than YOLO
    CONF_THRESHOLD = 0.3  # R-CNN training default
    IOU_THRESHOLD = 0.2   # R-CNN NMS threshold from training
    
    # Image size
    IMG_SIZE = 512
    
    def __init__(self):
        self.EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ====================================================================
# MODEL LOADER
# ====================================================================
def get_rcnn_model(num_classes=2):
    """Load the optimized Faster R-CNN model (same architecture as training)"""
    anchor_generator = AnchorGenerator(
        sizes=((4,), (8,), (16,), (32,), (64,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5
    )
    
    model = fasterrcnn_resnet50_fpn(
        weights=None,  # We'll load our trained weights
        trainable_backbone_layers=4,
        min_size=800,
        max_size=1333
    )
    
    model.rpn.anchor_generator = anchor_generator
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Set inference parameters
    model.roi_heads.nms_thresh = 0.2
    model.roi_heads.score_thresh = 0.01
    model.roi_heads.detections_per_img = 200
    
    return model


# ====================================================================
# EVALUATOR
# ====================================================================
class CandidateMiner:
    def __init__(self, config: EvalConfig):
        self.cfg = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"\n{'='*80}")
        print("🔬 FASTER R-CNN CANDIDATE MINING FOR H. PYLORI")
        print(f"{'='*80}")
        print(f"Device: {self.device}")
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
        
        print(f"\n🔥 Loading Model: {config.MODEL_PATH}")
        self.model = self._load_model()
        self.model.eval()
        
        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])
        
        self.excluded_files = self._get_training_files()
        
    def _load_model(self):
        """Load trained R-CNN model"""
        model = get_rcnn_model(num_classes=2)
        
        # Load checkpoint
        checkpoint = torch.load(self.cfg.MODEL_PATH, map_location=self.device)
        
        # Handle different checkpoint formats
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"✅ Model loaded from checkpoint")
            print(f"   Epoch: {checkpoint.get('epoch', 'N/A')}")
            print(f"   F1 Score: {checkpoint.get('f1', 'N/A'):.4f}")
            print(f"   Precision: {checkpoint.get('precision', 'N/A'):.4f}")
            print(f"   Recall: {checkpoint.get('recall', 'N/A'):.4f}")
        else:
            model.load_state_dict(checkpoint)
            print(f"✅ Model loaded (state dict only)")
        
        model.to(self.device)
        return model
        
    def _get_training_files(self):
        """Don't re-evaluate images the model was trained on"""
        train_imgs = list((self.cfg.TRAIN_OUTPUT_DIR / "images/train").glob("*"))
        val_imgs = list((self.cfg.TRAIN_OUTPUT_DIR / "images/val").glob("*"))
        
        # Handle symlinks - get the actual filename
        excluded = set()
        for p in train_imgs + val_imgs:
            if p.is_symlink():
                # Get the target filename
                excluded.add(p.name)
            else:
                excluded.add(p.name)
        
        print(f"🛑 Excluding {len(excluded)} patches already used in training.")
        return excluded

    def _batch_predict(self, image_list, desc):
        """Run inference safely in batches"""
        detections = []
        
        with torch.no_grad():
            for i in tqdm(range(0, len(image_list), self.cfg.BATCH_SIZE), desc=desc):
                batch_paths = image_list[i : i + self.cfg.BATCH_SIZE]
                
                # Load and preprocess images
                batch_tensors = []
                valid_paths = []
                
                for img_path in batch_paths:
                    try:
                        img = Image.open(img_path).convert("RGB")
                        img_tensor = self.transform(img)
                        batch_tensors.append(img_tensor.to(self.device))
                        valid_paths.append(img_path)
                    except Exception as e:
                        print(f"⚠️  Error loading {img_path.name}: {e}")
                        continue
                
                if not batch_tensors:
                    continue
                
                # Run R-CNN inference
                try:
                    predictions = self.model(batch_tensors)
                except RuntimeError as e:
                    print(f"⚠️  Inference error: {e}")
                    continue
                
                # Process Results
                for img_path, pred in zip(valid_paths, predictions):
                    boxes = pred['boxes'].cpu().numpy()
                    scores = pred['scores'].cpu().numpy()
                    
                    # Filter by confidence threshold
                    mask = scores >= self.cfg.CONF_THRESHOLD
                    boxes = boxes[mask]
                    scores = scores[mask]
                    
                    if len(boxes) > 0:
                        # Found something!
                        max_conf = float(scores.max())
                        box_count = len(boxes)
                        
                        detections.append({
                            'patch': img_path.name,
                            'path': img_path,
                            'max_conf': max_conf,
                            'count': box_count,
                            'boxes': boxes,
                            'scores': scores
                        })
                    
        return detections

    def run_mining_operation(self):
        print("\n🚀 STARTING R-CNN CANDIDATE MINING (ITERATION 3)")
        
        # 1. Collect all unverified images
        print("🔍 Scanning directories...")
        test_imgs = list(self.cfg.TEST_DATA_FULL.glob("*.png"))
        neg_imgs = list(self.cfg.ORIGINAL_NEG_IMG.glob("*.png"))
        
        all_candidates = test_imgs + neg_imgs
        
        print(f"   Test data images:     {len(test_imgs):,}")
        print(f"   Original negatives:   {len(neg_imgs):,}")
        print(f"   Total candidates:     {len(all_candidates):,}")
        
        # 2. Filter out training data
        clean_candidates = [p for p in all_candidates if p.name not in self.excluded_files]
        
        print(f"   After excluding training: {len(clean_candidates):,}")
        
        if not clean_candidates:
            print("❌ No images to check.")
            return

        # 3. Run Inference
        print(f"\n⚡ Running R-CNN Inference (Conf > {self.cfg.CONF_THRESHOLD})...")
        findings = self._batch_predict(clean_candidates, "Mining")
        
        # 4. Save & Report
        self._save_results(findings, len(clean_candidates))
        
        # 5. Generate XMLs for verification
        if findings:
            self._generate_verification_xmls(findings)

    def _save_results(self, findings, total_scanned):
        print("\n" + "="*80)
        print("📊 R-CNN MINING RESULTS")
        print("="*80)
        print(f"   Scanned:              {total_scanned:,}")
        print(f"   Found Candidates:     {len(findings):,} ({(len(findings)/max(total_scanned,1)*100):.1f}%)")
        
        if not findings:
            print("\n✅ No candidates found - all images appear negative!")
            return

        # Sort by confidence (highest first) -> These are best for verification
        findings.sort(key=lambda x: x['max_conf'], reverse=True)
        
        # Statistics
        conf_bins = {
            'Very High (>0.7)': sum(1 for f in findings if f['max_conf'] > 0.7),
            'High (0.5-0.7)': sum(1 for f in findings if 0.5 < f['max_conf'] <= 0.7),
            'Medium (0.3-0.5)': sum(1 for f in findings if 0.3 < f['max_conf'] <= 0.5),
            'Low (<0.3)': sum(1 for f in findings if f['max_conf'] <= 0.3),
        }
        
        print(f"\n📈 Confidence Distribution:")
        for category, count in conf_bins.items():
            if count > 0:
                print(f"   {category}: {count:,}")
        
        total_detections = sum(f['count'] for f in findings)
        print(f"\n🦠 Total Bacteria Detected: {total_detections:,}")
        print(f"   Avg per positive patch: {total_detections/len(findings):.1f}")
        
        # Save list for review tool / xml generation
        output_json = self.cfg.EVAL_OUTPUT_DIR / "rcnn_candidates_for_verification.json"
        
        serializable_findings = []
        for f in findings:
            serializable_findings.append({
                'patch': f['patch'],
                'path': str(f['path']),
                'max_conf': float(f['max_conf']),
                'count': int(f['count']),
                'boxes': f['boxes'].tolist(),
                'scores': f['scores'].tolist()
            })
            
        with open(output_json, 'w') as f:
            json.dump(serializable_findings, f, indent=2)
            
        print(f"\n💾 Candidate list saved: {output_json}")
        
        # Save Visualization of Top 20 Candidates
        vis_dir = self.cfg.EVAL_OUTPUT_DIR / "top_rcnn_candidates_visuals"
        vis_dir.mkdir(parents=True, exist_ok=True)
        
        print(f"🎨 Saving visuals for top 20 candidates...")
        for i, item in enumerate(findings[:20]):
            img = cv2.imread(str(item['path']))
            
            # Draw all boxes with confidence scores
            for box, score in zip(item['boxes'], item['scores']):
                x1, y1, x2, y2 = map(int, box)
                
                # Color based on confidence
                if score > 0.7:
                    color = (0, 255, 0)  # Green - very confident
                elif score > 0.5:
                    color = (0, 165, 255)  # Orange - confident
                else:
                    color = (0, 0, 255)  # Red - less confident
                
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                
                # Add confidence text
                label = f'{score:.2f}'
                (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(img, (x1, y1-20), (x1+w, y1), color, -1)
                cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX, 
                           0.5, (255, 255, 255), 1)
            
            # Add summary text
            summary = f"Rank {i+1} | Conf: {item['max_conf']:.3f} | Count: {item['count']}"
            cv2.putText(img, summary, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                       0.7, (255, 255, 255), 2)
            
            cv2.imwrite(str(vis_dir / f"rank_{i+1:02d}_{item['patch']}"), img)
        
        print(f"   Saved to: {vis_dir}")

    def _generate_verification_xmls(self, findings):
        """Generate ASAP XML files for pathologist verification"""
        print(f"\n📄 Generating ASAP XML files for verification...")
        
        xml_dir = self.cfg.EVAL_OUTPUT_DIR / "rcnn_verification_xmls"
        xml_dir.mkdir(parents=True, exist_ok=True)
        
        # Group findings by WSI (extract WSI ID from patch name)
        wsi_groups = defaultdict(list)
        
        for finding in findings:
            # Parse patch name: e.g., "593440_x61492_y24840.png"
            patch_name = finding['patch']
            parts = patch_name.replace('.png', '').split('_')
            
            if len(parts) >= 3:
                wsi_id = parts[0]
                try:
                    offset_x = int(parts[1].replace('x', ''))
                    offset_y = int(parts[2].replace('y', ''))
                    
                    wsi_groups[wsi_id].append({
                        'patch': patch_name,
                        'offset_x': offset_x,
                        'offset_y': offset_y,
                        'boxes': finding['boxes'],
                        'scores': finding['scores'],
                        'max_conf': finding['max_conf']
                    })
                except (ValueError, IndexError):
                    print(f"⚠️  Could not parse patch name: {patch_name}")
        
        print(f"   Found detections in {len(wsi_groups)} different WSIs")
        
        # Generate one XML per WSI
        for wsi_id, patches in tqdm(wsi_groups.items(), desc="   Generating XMLs"):
            xml_path = xml_dir / f"{wsi_id}_RCNN_candidates.xml"
            self._create_asap_xml(wsi_id, patches, xml_path)
        
        print(f"   XMLs saved to: {xml_dir}")
        print(f"\n✅ Generated {len(wsi_groups)} XML files for verification")

    def _create_asap_xml(self, wsi_id, patches, output_path):
        """Create ASAP-compatible XML annotation file"""
        # Create root element
        root = ET.Element('ASAP_Annotations')
        
        # Add annotations
        annotations = ET.SubElement(root, 'Annotations')
        
        annotation_id = 0
        
        for patch_data in patches:
            offset_x = patch_data['offset_x']
            offset_y = patch_data['offset_y']
            
            # Convert each detection box to WSI coordinates
            for box, score in zip(patch_data['boxes'], patch_data['scores']):
                x1, y1, x2, y2 = box
                
                # Convert to WSI coordinates
                wsi_x1 = offset_x + x1
                wsi_y1 = offset_y + y1
                wsi_x2 = offset_x + x2
                wsi_y2 = offset_y + y2
                
                # Create annotation
                annotation = ET.SubElement(annotations, 'Annotation')
                annotation.set('Name', f'RCNN_Detection')
                annotation.set('Type', 'Rectangle')
                annotation.set('PartOfGroup', 'RCNN_Candidates')
                annotation.set('Color', '#00FF00')  # Green
                
                # Add coordinates
                coordinates = ET.SubElement(annotation, 'Coordinates')
                
                # Rectangle: 4 corners
                corners = [
                    (wsi_x1, wsi_y1),
                    (wsi_x2, wsi_y1),
                    (wsi_x2, wsi_y2),
                    (wsi_x1, wsi_y2)
                ]
                
                for order, (x, y) in enumerate(corners):
                    coord = ET.SubElement(coordinates, 'Coordinate')
                    coord.set('Order', str(order))
                    coord.set('X', f'{x:.2f}')
                    coord.set('Y', f'{y:.2f}')
                
                annotation_id += 1
        
        # Add annotation groups
        groups = ET.SubElement(root, 'AnnotationGroups')
        group = ET.SubElement(groups, 'Group')
        group.set('Name', 'RCNN_Candidates')
        group.set('PartOfGroup', 'None')
        group.set('Color', '#00FF00')
        
        attributes = ET.SubElement(group, 'Attributes')
        
        # Pretty print and save
        xml_string = minidom.parseString(ET.tostring(root)).toprettyxml(indent="  ")
        
        with open(output_path, 'w') as f:
            f.write(xml_string)


# ====================================================================
# MAIN
# ====================================================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print("🔬 FASTER R-CNN EVALUATION & CANDIDATE MINING")
    print("="*80)
    
    cfg = EvalConfig()
    
    if not cfg.MODEL_PATH.exists():
        print(f"\n❌ Model not found: {cfg.MODEL_PATH}")
        print("   Please train the model first using train_rcnn_it3_final.py")
        exit(1)
    
    print(f"\n📋 Configuration:")
    print(f"   Model: {cfg.MODEL_PATH.name}")
    print(f"   Confidence Threshold: {cfg.CONF_THRESHOLD}")
    print(f"   IOU Threshold: {cfg.IOU_THRESHOLD}")
    print(f"   Batch Size: {cfg.BATCH_SIZE}")
    print(f"   Output: {cfg.EVAL_OUTPUT_DIR}")
    
    miner = CandidateMiner(cfg)
    miner.run_mining_operation()
    
    print("\n" + "="*80)
    print("✅ EVALUATION COMPLETE!")
    print("="*80)
    print(f"\n📁 Results saved in: {cfg.EVAL_OUTPUT_DIR}")
    print(f"\n📝 Next Steps:")
    print(f"   1. Review top candidates in: top_rcnn_candidates_visuals/")
    print(f"   2. Load XML files in ASAP for verification")
    print(f"   3. Mark true/false positives")
    print(f"   4. Use verified data for next iteration")
    print("="*80)


🔬 FASTER R-CNN EVALUATION & CANDIDATE MINING

📋 Configuration:
   Model: rcnn_bacteria_best_f1.pth
   Confidence Threshold: 0.3
   IOU Threshold: 0.2
   Batch Size: 16
   Output: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_rcnn_candidates_check

🔬 FASTER R-CNN CANDIDATE MINING FOR H. PYLORI
Device: cuda
GPU: Tesla V100-PCIE-32GB

🔥 Loading Model: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/models/rcnn_bacteria_best_f1.pth
✅ Model loaded from checkpoint
   Epoch: 42
   F1 Score: 0.5285
   Precision: 0.4924
   Recall: 0.5704
🛑 Excluding 5137 patches already used in training.

🚀 STARTING R-CNN CANDIDATE MINING (ITERATION 3)
🔍 Scanning directories...
   Test data images:     135,990
   Original negatives:   3,000
   Total candidates:     138,990
   After excluding training: 134,648

⚡ Running R-CNN Inference (Conf > 0.3)...


Mining: 100%|██████████████████████████████| 8416/8416 [1:14:27<00:00,  1.88it/s]



📊 R-CNN MINING RESULTS
   Scanned:              134,648
   Found Candidates:     3,985 (3.0%)

📈 Confidence Distribution:
   Very High (>0.7): 718
   High (0.5-0.7): 976
   Medium (0.3-0.5): 2,291

🦠 Total Bacteria Detected: 4,482
   Avg per positive patch: 1.1

💾 Candidate list saved: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_rcnn_candidates_check/rcnn_candidates_for_verification.json
🎨 Saving visuals for top 20 candidates...
   Saved to: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_rcnn_candidates_check/top_rcnn_candidates_visuals

📄 Generating ASAP XML files for verification...
   Found detections in 22 different WSIs


   Generating XMLs: 100%|████████████████████████| 22/22 [00:00<00:00, 24.32it/s]

   XMLs saved to: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_rcnn_candidates_check/rcnn_verification_xmls

✅ Generated 22 XML files for verification

✅ EVALUATION COMPLETE!

📁 Results saved in: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_rcnn_candidates_check

📝 Next Steps:
   1. Review top candidates in: top_rcnn_candidates_visuals/
   2. Load XML files in ASAP for verification
   3. Mark true/false positives
   4. Use verified data for next iteration


In [6]:
import cv2
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from collections import defaultdict
import xml.etree.ElementTree as ET
from xml.dom import minidom

# Import R-CNN model components
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision import transforms


# ====================================================================
# CONFIGURATION
# ====================================================================
class SingleWSITestConfig:
    PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
    
    # WSI to test
    WSI_ID = "522934"
    
    # Model
    TRAIN_OUTPUT_DIR = PROJECT_ROOT / "rcnn_optimal_final"
    MODEL_PATH = TRAIN_OUTPUT_DIR / "models/rcnn_bacteria_best_f1.pth"
    
    # Input patches location
    PATCHES_DIR = PROJECT_ROOT / "master-data/separated_patches/test_data_full/images"
    
    # Output for this single WSI test
    EVAL_OUTPUT_DIR = TRAIN_OUTPUT_DIR / f"evaluation_wsi_{WSI_ID}"
    
    # Inference Parameters
    BATCH_SIZE = 16
    CONF_THRESHOLD = 0.6
    IOU_THRESHOLD = 0.2
    
    # Image size
    IMG_SIZE = 512
    
    def __init__(self):
        self.EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ====================================================================
# MODEL LOADER
# ====================================================================
def get_rcnn_model(num_classes=2):
    """Load the optimized Faster R-CNN model"""
    anchor_generator = AnchorGenerator(
        sizes=((4,), (8,), (16,), (32,), (64,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5
    )
    
    model = fasterrcnn_resnet50_fpn(
        weights=None,
        trainable_backbone_layers=4,
        min_size=800,
        max_size=1333
    )
    
    model.rpn.anchor_generator = anchor_generator
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Set inference parameters
    model.roi_heads.nms_thresh = 0.2
    model.roi_heads.score_thresh = 0.01
    model.roi_heads.detections_per_img = 200
    
    return model


# ====================================================================
# SINGLE WSI TESTER
# ====================================================================
class SingleWSITester:
    def __init__(self, config: SingleWSITestConfig):
        self.cfg = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"\n{'='*80}")
        print(f"🔬 FASTER R-CNN SINGLE WSI TEST - WSI {config.WSI_ID}")
        print(f"{'='*80}")
        print(f"Device: {self.device}")
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
        
        print(f"\n🔥 Loading Model: {config.MODEL_PATH}")
        self.model = self._load_model()
        self.model.eval()
        
        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])
        
    def _load_model(self):
        """Load trained R-CNN model"""
        model = get_rcnn_model(num_classes=2)
        
        checkpoint = torch.load(self.cfg.MODEL_PATH, map_location=self.device)
        
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"✅ Model loaded from checkpoint")
            print(f"   Epoch: {checkpoint.get('epoch', 'N/A')}")
            print(f"   F1 Score: {checkpoint.get('f1', 'N/A'):.4f}")
            print(f"   Precision: {checkpoint.get('precision', 'N/A'):.4f}")
            print(f"   Recall: {checkpoint.get('recall', 'N/A'):.4f}")
        else:
            model.load_state_dict(checkpoint)
            print(f"✅ Model loaded (state dict only)")
        
        model.to(self.device)
        return model

    def _find_wsi_patches(self):
        """Find all patches belonging to the target WSI"""
        all_patches = list(self.cfg.PATCHES_DIR.glob("*.png"))
        wsi_patches = [p for p in all_patches if p.name.startswith(f"{self.cfg.WSI_ID}_")]
        
        print(f"\n🔍 Found {len(wsi_patches):,} patches for WSI {self.cfg.WSI_ID}")
        
        if len(wsi_patches) == 0:
            print(f"❌ No patches found with prefix '{self.cfg.WSI_ID}_' in {self.cfg.PATCHES_DIR}")
            return []
        
        # Sort for consistent processing
        wsi_patches.sort()
        
        return wsi_patches

    def _batch_predict(self, image_list):
        """Run inference in batches"""
        detections = []
        
        with torch.no_grad():
            for i in tqdm(range(0, len(image_list), self.cfg.BATCH_SIZE), desc="Running R-CNN"):
                batch_paths = image_list[i : i + self.cfg.BATCH_SIZE]
                
                batch_tensors = []
                valid_paths = []
                
                for img_path in batch_paths:
                    try:
                        img = Image.open(img_path).convert("RGB")
                        img_tensor = self.transform(img)
                        batch_tensors.append(img_tensor.to(self.device))
                        valid_paths.append(img_path)
                    except Exception as e:
                        print(f"⚠️  Error loading {img_path.name}: {e}")
                        continue
                
                if not batch_tensors:
                    continue
                
                try:
                    predictions = self.model(batch_tensors)
                except RuntimeError as e:
                    print(f"⚠️  Inference error: {e}")
                    continue
                
                # Process results
                for img_path, pred in zip(valid_paths, predictions):
                    boxes = pred['boxes'].cpu().numpy()
                    scores = pred['scores'].cpu().numpy()
                    
                    # Filter by confidence
                    mask = scores >= self.cfg.CONF_THRESHOLD
                    boxes = boxes[mask]
                    scores = scores[mask]
                    
                    # Parse patch coordinates from filename
                    # Format: 522934_x61492_y24840.png
                    parts = img_path.stem.split('_')
                    try:
                        offset_x = int(parts[1].replace('x', ''))
                        offset_y = int(parts[2].replace('y', ''))
                    except (ValueError, IndexError):
                        print(f"⚠️  Could not parse coordinates from: {img_path.name}")
                        continue
                    
                    if len(boxes) > 0:
                        detections.append({
                            'patch': img_path.name,
                            'path': img_path,
                            'offset_x': offset_x,
                            'offset_y': offset_y,
                            'boxes': boxes,
                            'scores': scores,
                            'max_conf': float(scores.max()),
                            'count': len(boxes)
                        })
                    
        return detections

    def run_test(self):
        """Execute single WSI test"""
        print(f"\n🚀 STARTING SINGLE WSI TEST - {self.cfg.WSI_ID}")
        
        # 1. Find patches
        patches = self._find_wsi_patches()
        if not patches:
            return
        
        # 2. Run inference
        print(f"\n⚡ Running R-CNN Inference (Conf > {self.cfg.CONF_THRESHOLD})...")
        detections = self._batch_predict(patches)
        
        # 3. Generate outputs
        self._save_results(detections, len(patches))
        
        if detections:
            # 4. Generate ASAP XML
            self._generate_asap_xml(detections)
            
            # 5. Create visualizations
            self._create_visualizations(detections)
        
        print(f"\n✅ Test complete for WSI {self.cfg.WSI_ID}")

    def _save_results(self, detections, total_patches):
        """Save detection results and statistics"""
        print("\n" + "="*80)
        print(f"📊 RESULTS FOR WSI {self.cfg.WSI_ID}")
        print("="*80)
        print(f"   Total patches scanned: {total_patches:,}")
        print(f"   Patches with detections: {len(detections):,} ({(len(detections)/max(total_patches,1)*100):.1f}%)")
        
        if not detections:
            print("\n✅ No bacteria detected in this WSI")
            
            # Save empty result
            output_json = self.cfg.EVAL_OUTPUT_DIR / f"{self.cfg.WSI_ID}_detections.json"
            with open(output_json, 'w') as f:
                json.dump({
                    'wsi_id': self.cfg.WSI_ID,
                    'total_patches': total_patches,
                    'patches_with_detections': 0,
                    'total_bacteria': 0,
                    'detections': []
                }, f, indent=2)
            return
        
        # Calculate statistics
        total_bacteria = sum(d['count'] for d in detections)
        avg_per_patch = total_bacteria / len(detections)
        max_conf = max(d['max_conf'] for d in detections)
        avg_conf = np.mean([d['max_conf'] for d in detections])
        
        print(f"\n🦠 Detection Statistics:")
        print(f"   Total bacteria detected: {total_bacteria:,}")
        print(f"   Avg per positive patch: {avg_per_patch:.1f}")
        print(f"   Max confidence: {max_conf:.3f}")
        print(f"   Avg confidence: {avg_conf:.3f}")
        
        # Confidence distribution
        conf_bins = {
            'Very High (>0.7)': sum(1 for d in detections if d['max_conf'] > 0.7),
            'High (0.5-0.7)': sum(1 for d in detections if 0.5 < d['max_conf'] <= 0.7),
            'Medium (0.3-0.5)': sum(1 for d in detections if 0.3 < d['max_conf'] <= 0.5),
        }
        
        print(f"\n📈 Confidence Distribution:")
        for category, count in conf_bins.items():
            if count > 0:
                print(f"   {category}: {count:,} patches")
        
        # Save JSON
        output_json = self.cfg.EVAL_OUTPUT_DIR / f"{self.cfg.WSI_ID}_detections.json"
        
        serializable_detections = []
        for d in detections:
            serializable_detections.append({
                'patch': d['patch'],
                'offset_x': d['offset_x'],
                'offset_y': d['offset_y'],
                'boxes': d['boxes'].tolist(),
                'scores': d['scores'].tolist(),
                'max_conf': d['max_conf'],
                'count': d['count']
            })
        
        result_data = {
            'wsi_id': self.cfg.WSI_ID,
            'total_patches': total_patches,
            'patches_with_detections': len(detections),
            'total_bacteria': total_bacteria,
            'max_confidence': float(max_conf),
            'avg_confidence': float(avg_conf),
            'detections': serializable_detections
        }
        
        with open(output_json, 'w') as f:
            json.dump(result_data, f, indent=2)
        
        print(f"\n💾 Results saved: {output_json}")

    def _generate_asap_xml(self, detections):
        """Generate ASAP XML annotation file"""
        print(f"\n📄 Generating ASAP XML...")
        
        # Create root
        root = ET.Element('ASAP_Annotations')
        
        # Add annotations
        annotations = ET.SubElement(root, 'Annotations')
        
        annotation_id = 0
        
        for detection in detections:
            offset_x = detection['offset_x']
            offset_y = detection['offset_y']
            
            # Convert each box to WSI coordinates
            for box, score in zip(detection['boxes'], detection['scores']):
                x1, y1, x2, y2 = box
                
                # WSI coordinates
                wsi_x1 = offset_x + x1
                wsi_y1 = offset_y + y1
                wsi_x2 = offset_x + x2
                wsi_y2 = offset_y + y2
                
                # Create annotation
                annotation = ET.SubElement(annotations, 'Annotation')
                annotation.set('Name', f'Bacteria')
                annotation.set('Type', 'Rectangle')
                annotation.set('PartOfGroup', 'H_pylori')
                annotation.set('Color', '#00FF00')
                
                # Add coordinates (rectangle = 4 corners)
                coordinates = ET.SubElement(annotation, 'Coordinates')
                
                corners = [
                    (wsi_x1, wsi_y1),
                    (wsi_x2, wsi_y1),
                    (wsi_x2, wsi_y2),
                    (wsi_x1, wsi_y2)
                ]
                
                for order, (x, y) in enumerate(corners):
                    coord = ET.SubElement(coordinates, 'Coordinate')
                    coord.set('Order', str(order))
                    coord.set('X', f'{x:.2f}')
                    coord.set('Y', f'{y:.2f}')
                
                annotation_id += 1
        
        # Add annotation groups
        groups = ET.SubElement(root, 'AnnotationGroups')
        group = ET.SubElement(groups, 'Group')
        group.set('Name', 'H_pylori')
        group.set('PartOfGroup', 'None')
        group.set('Color', '#00FF00')
        
        attributes = ET.SubElement(group, 'Attributes')
        
        # Pretty print
        xml_string = minidom.parseString(ET.tostring(root)).toprettyxml(indent="  ")
        
        # Save XML
        xml_path = self.cfg.EVAL_OUTPUT_DIR / f"{self.cfg.WSI_ID}.xml"
        with open(xml_path, 'w') as f:
            f.write(xml_string)
        
        print(f"   ✅ XML saved: {xml_path}")
        print(f"   📊 Total annotations: {annotation_id:,}")

    def _create_visualizations(self, detections):
        """Create visual outputs"""
        print(f"\n🎨 Creating visualizations...")
        
        vis_dir = self.cfg.EVAL_OUTPUT_DIR / "visualizations"
        vis_dir.mkdir(exist_ok=True)
        
        # Sort by confidence for top detections
        sorted_detections = sorted(detections, key=lambda x: x['max_conf'], reverse=True)
        
        # Save top 20 detections
        num_to_save = min(20, len(sorted_detections))
        
        for i, detection in enumerate(sorted_detections[:num_to_save]):
            img = cv2.imread(str(detection['path']))
            
            # Draw all boxes
            for box, score in zip(detection['boxes'], detection['scores']):
                x1, y1, x2, y2 = map(int, box)
                
                # Color based on confidence
                if score > 0.7:
                    color = (0, 255, 0)  # Green
                elif score > 0.5:
                    color = (0, 165, 255)  # Orange
                else:
                    color = (0, 0, 255)  # Red
                
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                
                # Confidence label
                label = f'{score:.2f}'
                (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(img, (x1, y1-20), (x1+w, y1), color, -1)
                cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX,
                           0.5, (255, 255, 255), 1)
            
            # Summary
            summary = f"Rank {i+1} | Conf: {detection['max_conf']:.3f} | Count: {detection['count']}"
            cv2.putText(img, summary, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                       0.7, (255, 255, 255), 2)
            
            output_name = f"rank_{i+1:02d}_{detection['patch']}"
            cv2.imwrite(str(vis_dir / output_name), img)
        
        print(f"   ✅ Saved {num_to_save} visualizations to: {vis_dir}")
        
        # Create summary figure
        self._create_summary_figure(sorted_detections[:20], vis_dir)

    def _create_summary_figure(self, top_detections, vis_dir):
        """Create a grid summary of top detections"""
        if not top_detections:
            return
        
        num_imgs = min(9, len(top_detections))
        if num_imgs == 0:
            return
        
        fig, axes = plt.subplots(3, 3, figsize=(15, 15))
        axes = axes.flatten()
        
        for i in range(9):
            if i < num_imgs:
                detection = top_detections[i]
                
                # Load image
                img = cv2.imread(str(detection['path']))
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                # Draw boxes
                for box, score in zip(detection['boxes'], detection['scores']):
                    x1, y1, x2, y2 = map(int, box)
                    color = (0, 255, 0) if score > 0.7 else (255, 165, 0) if score > 0.5 else (255, 0, 0)
                    cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                
                axes[i].imshow(img)
                axes[i].set_title(f"Rank {i+1}\nConf: {detection['max_conf']:.3f} | Count: {detection['count']}", 
                                 fontsize=10)
                axes[i].axis('off')
            else:
                axes[i].axis('off')
        
        plt.suptitle(f"WSI {self.cfg.WSI_ID} - Top 9 Detections", fontsize=16, fontweight='bold')
        plt.tight_layout()
        
        summary_path = vis_dir / f"{self.cfg.WSI_ID}_summary.png"
        plt.savefig(summary_path, dpi=150, bbox_inches='tight')
        plt.close()
        
        print(f"   ✅ Summary figure saved: {summary_path}")


# ====================================================================
# MAIN
# ====================================================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print("🔬 FASTER R-CNN SINGLE WSI TEST")
    print("="*80)
    
    cfg = SingleWSITestConfig()
    
    # Validation
    if not cfg.MODEL_PATH.exists():
        print(f"\n❌ Model not found: {cfg.MODEL_PATH}")
        exit(1)
    
    if not cfg.PATCHES_DIR.exists():
        print(f"\n❌ Patches directory not found: {cfg.PATCHES_DIR}")
        exit(1)
    
    print(f"\n📋 Configuration:")
    print(f"   WSI ID: {cfg.WSI_ID}")
    print(f"   Model: {cfg.MODEL_PATH.name}")
    print(f"   Patches location: {cfg.PATCHES_DIR}")
    print(f"   Confidence threshold: {cfg.CONF_THRESHOLD}")
    print(f"   Output directory: {cfg.EVAL_OUTPUT_DIR}")
    
    # Run test
    tester = SingleWSITester(cfg)
    tester.run_test()
    
    print("\n" + "="*80)
    print("✅ SINGLE WSI TEST COMPLETE!")
    print("="*80)
    print(f"\n📁 Results saved in: {cfg.EVAL_OUTPUT_DIR}")
    print(f"\n📝 Generated files:")
    print(f"   • {cfg.WSI_ID}.xml - ASAP annotation file")
    print(f"   • {cfg.WSI_ID}_detections.json - Detection details")
    print(f"   • visualizations/ - Top detection images")
    print("="*80)


🔬 FASTER R-CNN SINGLE WSI TEST

📋 Configuration:
   WSI ID: 522934
   Model: rcnn_bacteria_best_f1.pth
   Patches location: /home/biopsy_gregorova/hpylori_project/master-data/separated_patches/test_data_full/images
   Confidence threshold: 0.6
   Output directory: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934

🔬 FASTER R-CNN SINGLE WSI TEST - WSI 522934
Device: cuda
GPU: Tesla V100-PCIE-32GB

🔥 Loading Model: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/models/rcnn_bacteria_best_f1.pth
✅ Model loaded from checkpoint
   Epoch: 42
   F1 Score: 0.5285
   Precision: 0.4924
   Recall: 0.5704

🚀 STARTING SINGLE WSI TEST - 522934

🔍 Found 17,670 patches for WSI 522934

⚡ Running R-CNN Inference (Conf > 0.6)...


Running R-CNN: 100%|█████████████████████████| 1105/1105 [09:37<00:00,  1.91it/s]



📊 RESULTS FOR WSI 522934
   Total patches scanned: 17,670
   Patches with detections: 582 (3.3%)

🦠 Detection Statistics:
   Total bacteria detected: 647
   Avg per positive patch: 1.1
   Max confidence: 0.961
   Avg confidence: 0.788

📈 Confidence Distribution:
   Very High (>0.7): 448 patches
   High (0.5-0.7): 134 patches

💾 Results saved: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/522934_detections.json

📄 Generating ASAP XML...
   ✅ XML saved: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/522934.xml
   📊 Total annotations: 647

🎨 Creating visualizations...
   ✅ Saved 20 visualizations to: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/visualizations
   ✅ Summary figure saved: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/visualizations/522934_summary.png

✅ Test complete for WSI 522934

✅ SINGLE WSI TEST COMPLETE!

📁 Results saved in: /home/bio

In [10]:
import cv2
import numpy as np
import torch
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import json
from pathlib import Path
from tqdm import tqdm
from PIL import Image
from collections import defaultdict
import xml.etree.ElementTree as ET
from xml.dom import minidom

# Import R-CNN model components
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision import transforms
from torchvision.ops import nms


# ====================================================================
# CONFIGURATION
# ====================================================================
class SingleWSITestConfig:
    PROJECT_ROOT = Path("/home/biopsy_gregorova/hpylori_project")
    
    # WSI to test
    WSI_ID = "522934"
    
    # Model
    TRAIN_OUTPUT_DIR = PROJECT_ROOT / "rcnn_optimal_final"
    MODEL_PATH = TRAIN_OUTPUT_DIR / "models/rcnn_bacteria_best_f1.pth"
    
    # Input patches location
    PATCHES_DIR = PROJECT_ROOT / "master-data/separated_patches/test_data_full/images"
    
    # Output for this single WSI test
    EVAL_OUTPUT_DIR = TRAIN_OUTPUT_DIR / f"evaluation_wsi_{WSI_ID}"
    
    # Inference Parameters
    BATCH_SIZE = 16
    CONF_THRESHOLD = 0.3
    IOU_THRESHOLD = 0.2  # NMS threshold for overlapping boxes (20% - more aggressive)
    
    # Image size
    IMG_SIZE = 512
    
    def __init__(self):
        self.EVAL_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ====================================================================
# HELPER FUNCTIONS
# ====================================================================
def calculate_iou(box1, box2):
    """Calculate IoU between two boxes [x1, y1, x2, y2]"""
    x1_inter = max(box1[0], box2[0])
    y1_inter = max(box1[1], box2[1])
    x2_inter = min(box1[2], box2[2])
    y2_inter = min(box1[3], box2[3])
    
    if x2_inter < x1_inter or y2_inter < y1_inter:
        return 0.0
    
    inter_area = (x2_inter - x1_inter) * (y2_inter - y1_inter)
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    
    union_area = box1_area + box2_area - inter_area
    
    if union_area == 0:
        return 0.0
    
    return inter_area / union_area


def apply_nms_to_wsi_boxes(all_boxes, all_scores, iou_threshold=0.2):
    """
    Apply NMS to all boxes in WSI coordinates
    Keep the box with highest confidence when overlap > threshold
    """
    if len(all_boxes) == 0:
        return np.array([]), np.array([])
    
    # Convert to tensor for NMS
    boxes_tensor = torch.from_numpy(all_boxes).float()
    scores_tensor = torch.from_numpy(all_scores).float()
    
    # Apply PyTorch's NMS (well-tested and efficient)
    keep_indices = nms(boxes_tensor, scores_tensor, iou_threshold)
    
    # Return filtered boxes and scores
    kept_boxes = all_boxes[keep_indices.numpy()]
    kept_scores = all_scores[keep_indices.numpy()]
    
    removed_count = len(all_boxes) - len(kept_boxes)
    if removed_count > 0:
        print(f"   🗑️  NMS removed {removed_count} overlapping boxes")
    
    return kept_boxes, kept_scores


# ====================================================================
# MODEL LOADER
# ====================================================================
def get_rcnn_model(num_classes=2):
    """Load the optimized Faster R-CNN model"""
    anchor_generator = AnchorGenerator(
        sizes=((4,), (8,), (16,), (32,), (64,)),
        aspect_ratios=((0.5, 1.0, 2.0),) * 5
    )
    
    model = fasterrcnn_resnet50_fpn(
        weights=None,
        trainable_backbone_layers=4,
        min_size=800,
        max_size=1333
    )
    
    model.rpn.anchor_generator = anchor_generator
    
    in_features = model.roi_heads.box_predictor.cls_score.in_features
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)
    
    # Set inference parameters
    model.roi_heads.nms_thresh = 0.2
    model.roi_heads.score_thresh = 0.01
    model.roi_heads.detections_per_img = 200
    
    return model


# ====================================================================
# SINGLE WSI TESTER
# ====================================================================
class SingleWSITester:
    def __init__(self, config: SingleWSITestConfig):
        self.cfg = config
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        
        print(f"\n{'='*80}")
        print(f"🔬 FASTER R-CNN SINGLE WSI TEST - WSI {config.WSI_ID}")
        print(f"{'='*80}")
        print(f"Device: {self.device}")
        if torch.cuda.is_available():
            print(f"GPU: {torch.cuda.get_device_name(0)}")
        
        print(f"\n🔥 Loading Model: {config.MODEL_PATH}")
        self.model = self._load_model()
        self.model.eval()
        
        self.transform = transforms.Compose([
            transforms.ToTensor()
        ])
        
    def _load_model(self):
        """Load trained R-CNN model"""
        model = get_rcnn_model(num_classes=2)
        
        checkpoint = torch.load(self.cfg.MODEL_PATH, map_location=self.device)
        
        if 'model_state_dict' in checkpoint:
            model.load_state_dict(checkpoint['model_state_dict'])
            print(f"✅ Model loaded from checkpoint")
            print(f"   Epoch: {checkpoint.get('epoch', 'N/A')}")
            print(f"   F1 Score: {checkpoint.get('f1', 'N/A'):.4f}")
            print(f"   Precision: {checkpoint.get('precision', 'N/A'):.4f}")
            print(f"   Recall: {checkpoint.get('recall', 'N/A'):.4f}")
        else:
            model.load_state_dict(checkpoint)
            print(f"✅ Model loaded (state dict only)")
        
        model.to(self.device)
        return model

    def _find_wsi_patches(self):
        """Find all patches belonging to the target WSI"""
        all_patches = list(self.cfg.PATCHES_DIR.glob("*.png"))
        wsi_patches = [p for p in all_patches if p.name.startswith(f"{self.cfg.WSI_ID}_")]
        
        print(f"\n🔍 Found {len(wsi_patches):,} patches for WSI {self.cfg.WSI_ID}")
        
        if len(wsi_patches) == 0:
            print(f"❌ No patches found with prefix '{self.cfg.WSI_ID}_' in {self.cfg.PATCHES_DIR}")
            return []
        
        # Sort for consistent processing
        wsi_patches.sort()
        
        return wsi_patches

    def _batch_predict(self, image_list):
        """Run inference in batches and collect all detections with WSI coordinates"""
        all_detections = []
        
        with torch.no_grad():
            for i in tqdm(range(0, len(image_list), self.cfg.BATCH_SIZE), desc="Running R-CNN"):
                batch_paths = image_list[i : i + self.cfg.BATCH_SIZE]
                
                batch_tensors = []
                valid_paths = []
                
                for img_path in batch_paths:
                    try:
                        img = Image.open(img_path).convert("RGB")
                        img_tensor = self.transform(img)
                        batch_tensors.append(img_tensor.to(self.device))
                        valid_paths.append(img_path)
                    except Exception as e:
                        print(f"⚠️  Error loading {img_path.name}: {e}")
                        continue
                
                if not batch_tensors:
                    continue
                
                try:
                    predictions = self.model(batch_tensors)
                except RuntimeError as e:
                    print(f"⚠️  Inference error: {e}")
                    continue
                
                # Process results - convert to WSI coordinates immediately
                for img_path, pred in zip(valid_paths, predictions):
                    boxes = pred['boxes'].cpu().numpy()
                    scores = pred['scores'].cpu().numpy()
                    
                    # Filter by confidence
                    mask = scores >= self.cfg.CONF_THRESHOLD
                    boxes = boxes[mask]
                    scores = scores[mask]
                    
                    if len(boxes) == 0:
                        continue
                    
                    # Parse patch coordinates from filename
                    # Format: 522934_x61492_y24840.png
                    parts = img_path.stem.split('_')
                    try:
                        offset_x = int(parts[1].replace('x', ''))
                        offset_y = int(parts[2].replace('y', ''))
                    except (ValueError, IndexError):
                        print(f"⚠️  Could not parse coordinates from: {img_path.name}")
                        continue
                    
                    # Convert all boxes to WSI coordinates
                    for box, score in zip(boxes, scores):
                        x1, y1, x2, y2 = box
                        
                        # Don't clamp to patch boundaries - keep full box dimensions
                        # The model predicts the full bacteria, even if it extends beyond patch edge
                        
                        # Skip invalid boxes (too small or inverted)
                        if x2 - x1 < 5 or y2 - y1 < 5:
                            continue
                        
                        # Convert to WSI coordinates using full box dimensions
                        # Use float coordinates for accurate conversion
                        wsi_box = np.array([
                            float(offset_x + x1),
                            float(offset_y + y1),
                            float(offset_x + x2),
                            float(offset_y + y2)
                        ])
                        
                        all_detections.append({
                            'patch': img_path.name,
                            'patch_box': np.array([x1, y1, x2, y2]),  # Original patch coordinates
                            'wsi_box': wsi_box,  # WSI coordinates (full box)
                            'score': float(score),
                            'offset_x': offset_x,
                            'offset_y': offset_y
                        })
                    
        return all_detections

    def run_test(self):
        """Execute single WSI test with NMS post-processing"""
        print(f"\n🚀 STARTING SINGLE WSI TEST - {self.cfg.WSI_ID}")
        
        # 1. Find patches
        patches = self._find_wsi_patches()
        if not patches:
            return
        
        # 2. Run inference - get all detections in WSI coordinates
        print(f"\n⚡ Running R-CNN Inference (Conf > {self.cfg.CONF_THRESHOLD})...")
        all_detections = self._batch_predict(patches)
        
        print(f"\n📊 Initial detections: {len(all_detections):,}")
        
        if not all_detections:
            print("✅ No bacteria detected in this WSI")
            self._save_empty_results(len(patches))
            return
        
        # Debug: Check for potential duplicates
        print(f"\n🔍 Analyzing detections before NMS...")
        unique_patches = set([d['patch'] for d in all_detections])
        print(f"   Detections from {len(unique_patches)} unique patches")
        print(f"   Avg detections per patch: {len(all_detections)/len(unique_patches):.1f}")
        
        # 3. Apply NMS to remove overlapping boxes
        print(f"🔧 Applying NMS (IoU threshold: {self.cfg.IOU_THRESHOLD})...")
        
        # Extract WSI boxes and scores
        wsi_boxes = np.array([d['wsi_box'] for d in all_detections])
        scores = np.array([d['score'] for d in all_detections])
        
        print(f"   Before NMS: {len(all_detections):,} boxes")
        print(f"   Score range: {scores.min():.3f} - {scores.max():.3f}")
        
        # Apply NMS
        kept_boxes, kept_scores = apply_nms_to_wsi_boxes(
            wsi_boxes, 
            scores, 
            self.cfg.IOU_THRESHOLD
        )
        
        print(f"   After NMS:  {len(kept_boxes):,} boxes")
        print(f"   Removed:    {len(all_detections) - len(kept_boxes):,} overlapping boxes ({(len(all_detections) - len(kept_boxes))/max(len(all_detections),1)*100:.1f}%)")
        
        if len(kept_boxes) < len(all_detections):
            print(f"   ✅ NMS successfully removed overlaps")
        
        # 4. Create final detection list
        final_detections = []
        for box, score in zip(kept_boxes, kept_scores):
            final_detections.append({
                'wsi_box': box,
                'score': float(score)
            })
        
        # Sort by confidence
        final_detections.sort(key=lambda x: x['score'], reverse=True)
        
        # 5. Generate outputs
        self._save_results(final_detections, len(patches))
        self._generate_imagescope_xml(final_detections)
        self._create_visualizations(patches, final_detections)
        
        print(f"\n✅ Test complete for WSI {self.cfg.WSI_ID}")

    def _save_empty_results(self, total_patches):
        """Save empty result when no detections found"""
        output_json = self.cfg.EVAL_OUTPUT_DIR / f"{self.cfg.WSI_ID}_detections.json"
        with open(output_json, 'w') as f:
            json.dump({
                'wsi_id': self.cfg.WSI_ID,
                'total_patches': total_patches,
                'total_bacteria': 0,
                'detections': []
            }, f, indent=2)

    def _save_results(self, detections, total_patches):
        """Save detection results and statistics"""
        print("\n" + "="*80)
        print(f"📊 FINAL RESULTS FOR WSI {self.cfg.WSI_ID}")
        print("="*80)
        print(f"   Total patches scanned: {total_patches:,}")
        print(f"   Total bacteria detected: {len(detections):,}")
        
        if not detections:
            return
        
        # Calculate statistics
        max_conf = max(d['score'] for d in detections)
        avg_conf = np.mean([d['score'] for d in detections])
        
        print(f"\n🦠 Detection Statistics:")
        print(f"   Max confidence: {max_conf:.3f}")
        print(f"   Avg confidence: {avg_conf:.3f}")
        
        # Confidence distribution
        conf_bins = {
            'Very High (>0.7)': sum(1 for d in detections if d['score'] > 0.7),
            'High (0.5-0.7)': sum(1 for d in detections if 0.5 < d['score'] <= 0.7),
            'Medium (0.3-0.5)': sum(1 for d in detections if 0.3 < d['score'] <= 0.5),
        }
        
        print(f"\n📈 Confidence Distribution:")
        for category, count in conf_bins.items():
            if count > 0:
                print(f"   {category}: {count:,} bacteria")
        
        # Save JSON
        output_json = self.cfg.EVAL_OUTPUT_DIR / f"{self.cfg.WSI_ID}_detections.json"
        
        serializable_detections = []
        for d in detections:
            serializable_detections.append({
                'wsi_box': d['wsi_box'].tolist(),
                'score': d['score']
            })
        
        result_data = {
            'wsi_id': self.cfg.WSI_ID,
            'total_patches': total_patches,
            'total_bacteria': len(detections),
            'max_confidence': float(max_conf),
            'avg_confidence': float(avg_conf),
            'nms_iou_threshold': self.cfg.IOU_THRESHOLD,
            'detections': serializable_detections
        }
        
        with open(output_json, 'w') as f:
            json.dump(result_data, f, indent=2)
        
        print(f"\n💾 Results saved: {output_json}")

    def _generate_imagescope_xml(self, detections):
        """Generate ImageScope XML annotation file"""
        print(f"\n📄 Generating ImageScope XML...")
        
        # Create root with MicronsPerPixel attribute
        root = ET.Element('Annotations')
        root.set('MicronsPerPixel', '0.253200')
        
        # Create main annotation container
        annotation_container = ET.SubElement(root, 'Annotation')
        annotation_container.set('Id', '1')
        annotation_container.set('Name', 'RCNN_Detection')
        annotation_container.set('ReadOnly', '0')
        annotation_container.set('NameReadOnly', '0')
        annotation_container.set('LineColorReadOnly', '0')
        annotation_container.set('Incremental', '0')
        annotation_container.set('Type', '4')
        annotation_container.set('LineColor', '65280')  # Green in decimal
        annotation_container.set('Visible', '1')
        annotation_container.set('Selected', '1')
        annotation_container.set('MarkupImagePath', '')
        annotation_container.set('MacroName', '')
        
        # Add empty Attributes
        ET.SubElement(annotation_container, 'Attributes')
        
        # Create Regions container
        regions = ET.SubElement(annotation_container, 'Regions')
        ET.SubElement(regions, 'RegionAttributeHeaders')
        
        region_id = 1
        
        for detection in detections:
            box = detection['wsi_box']
            score = detection['score']
            
            # WSI coordinates (already in WSI space)
            # Round to nearest integer to avoid truncation issues
            x1, y1, x2, y2 = box
            wsi_x1 = round(x1)
            wsi_y1 = round(y1)
            wsi_x2 = round(x2)
            wsi_y2 = round(y2)
            
            # Create region
            region = ET.SubElement(regions, 'Region')
            region.set('Id', str(region_id))
            region.set('Type', '0')
            region.set('Zoom', '1')
            region.set('Selected', '0')
            region.set('ImageLocation', '')
            region.set('ImageFocus', '-1')
            region.set('Length', '0')
            region.set('Area', '0')
            region.set('LengthMicrons', '0')
            region.set('AreaMicrons', '0')
            region.set('Text', f'{score:.2f}')  # Confidence score
            region.set('NegativeROA', '0')
            region.set('InputRegionId', '0')
            region.set('Analyze', '1')
            region.set('DisplayId', str(region_id))
            
            # Add empty Attributes
            ET.SubElement(region, 'Attributes')
            
            # Add Vertices (rectangle = 4 corners)
            vertices = ET.SubElement(region, 'Vertices')
            
            corners = [
                (wsi_x1, wsi_y1),
                (wsi_x2, wsi_y1),
                (wsi_x2, wsi_y2),
                (wsi_x1, wsi_y2)
            ]
            
            for x, y in corners:
                vertex = ET.SubElement(vertices, 'Vertex')
                vertex.set('X', str(x))
                vertex.set('Y', str(y))
                vertex.set('Z', '0')
            
            region_id += 1
        
        # Pretty print
        xml_string = minidom.parseString(ET.tostring(root)).toprettyxml(indent="\t")
        
        # Save XML
        xml_path = self.cfg.EVAL_OUTPUT_DIR / f"{self.cfg.WSI_ID}.xml"
        with open(xml_path, 'w') as f:
            f.write(xml_string)
        
        print(f"   ✅ XML saved: {xml_path}")
        print(f"   📊 Total annotations: {region_id - 1:,}")

    def _create_visualizations(self, patches, detections):
        """Create visual outputs showing detections on patches"""
        print(f"\n🎨 Creating visualizations...")
        
        vis_dir = self.cfg.EVAL_OUTPUT_DIR / "visualizations"
        vis_dir.mkdir(exist_ok=True)
        
        # Group detections by patch for visualization
        patch_detections = defaultdict(list)
        
        for detection in detections:
            box = detection['wsi_box']
            score = detection['score']
            
            # Find which patch(es) this detection overlaps with
            for patch_path in patches:
                parts = patch_path.stem.split('_')
                try:
                    offset_x = int(parts[1].replace('x', ''))
                    offset_y = int(parts[2].replace('y', ''))
                except (ValueError, IndexError):
                    continue
                
                # Check if detection overlaps with this patch
                # A detection overlaps if any part of it is within the patch bounds
                patch_x1 = offset_x
                patch_y1 = offset_y
                patch_x2 = offset_x + self.cfg.IMG_SIZE
                patch_y2 = offset_y + self.cfg.IMG_SIZE
                
                # Check for overlap
                if not (box[2] < patch_x1 or box[0] > patch_x2 or
                        box[3] < patch_y1 or box[1] > patch_y2):
                    
                    # Convert to patch coordinates and clamp to patch bounds
                    patch_box = [
                        max(0, box[0] - offset_x),
                        max(0, box[1] - offset_y),
                        min(self.cfg.IMG_SIZE, box[2] - offset_x),
                        min(self.cfg.IMG_SIZE, box[3] - offset_y)
                    ]
                    
                    # Only add if the box has some visible area in this patch
                    if patch_box[2] - patch_box[0] > 2 and patch_box[3] - patch_box[1] > 2:
                        patch_detections[patch_path.name].append({
                            'box': patch_box,
                            'score': score
                        })
        
        # Save visualizations for patches with detections
        print(f"   Visualizing {len(patch_detections)} patches with detections...")
        
        sorted_patches = sorted(
            patch_detections.items(), 
            key=lambda x: max([d['score'] for d in x[1]]), 
            reverse=True
        )
        
        num_to_save = min(20, len(sorted_patches))
        
        for i, (patch_name, dets) in enumerate(sorted_patches[:num_to_save]):
            patch_path = self.cfg.PATCHES_DIR / patch_name
            
            if not patch_path.exists():
                continue
            
            img = cv2.imread(str(patch_path))
            
            # Draw all boxes
            for det in dets:
                box = det['box']
                score = det['score']
                
                x1, y1, x2, y2 = map(int, box)
                
                # Ensure coordinates are within image bounds
                x1 = max(0, min(x1, img.shape[1]))
                y1 = max(0, min(y1, img.shape[0]))
                x2 = max(0, min(x2, img.shape[1]))
                y2 = max(0, min(y2, img.shape[0]))
                
                # Color based on confidence
                if score > 0.7:
                    color = (0, 255, 0)  # Green
                elif score > 0.5:
                    color = (0, 165, 255)  # Orange
                else:
                    color = (0, 0, 255)  # Red
                
                cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
                
                # Confidence label
                label = f'{score:.2f}'
                (w, h), _ = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.5, 1)
                cv2.rectangle(img, (x1, y1-20), (x1+w, y1), color, -1)
                cv2.putText(img, label, (x1, y1-5), cv2.FONT_HERSHEY_SIMPLEX,
                           0.5, (255, 255, 255), 1)
            
            # Summary
            max_score = max([d['score'] for d in dets])
            summary = f"Rank {i+1} | Max Conf: {max_score:.3f} | Count: {len(dets)}"
            cv2.putText(img, summary, (10, 30), cv2.FONT_HERSHEY_SIMPLEX,
                       0.7, (255, 255, 255), 2)
            
            output_name = f"rank_{i+1:02d}_{patch_name}"
            cv2.imwrite(str(vis_dir / output_name), img)
        
        print(f"   ✅ Saved {num_to_save} visualizations to: {vis_dir}")


# ====================================================================
# MAIN
# ====================================================================
if __name__ == "__main__":
    print("\n" + "="*80)
    print("🔬 FASTER R-CNN SINGLE WSI TEST WITH NMS")
    print("="*80)
    
    cfg = SingleWSITestConfig()
    
    # Validation
    if not cfg.MODEL_PATH.exists():
        print(f"\n❌ Model not found: {cfg.MODEL_PATH}")
        exit(1)
    
    if not cfg.PATCHES_DIR.exists():
        print(f"\n❌ Patches directory not found: {cfg.PATCHES_DIR}")
        exit(1)
    
    print(f"\n📋 Configuration:")
    print(f"   WSI ID: {cfg.WSI_ID}")
    print(f"   Model: {cfg.MODEL_PATH.name}")
    print(f"   Patches location: {cfg.PATCHES_DIR}")
    print(f"   Confidence threshold: {cfg.CONF_THRESHOLD}")
    print(f"   NMS IoU threshold: {cfg.IOU_THRESHOLD} (20% overlap - aggressive)")
    print(f"   Output directory: {cfg.EVAL_OUTPUT_DIR}")
    
    # Run test
    tester = SingleWSITester(cfg)
    tester.run_test()
    
    print("\n" + "="*80)
    print("✅ SINGLE WSI TEST COMPLETE!")
    print("="*80)
    print(f"\n📁 Results saved in: {cfg.EVAL_OUTPUT_DIR}")
    print(f"\n📝 Generated files:")
    print(f"   • {cfg.WSI_ID}.xml - ImageScope annotation file")
    print(f"   • {cfg.WSI_ID}_detections.json - Detection details")
    print(f"   • visualizations/ - Top detection images")
    print("="*80)


🔬 FASTER R-CNN SINGLE WSI TEST WITH NMS

📋 Configuration:
   WSI ID: 522934
   Model: rcnn_bacteria_best_f1.pth
   Patches location: /home/biopsy_gregorova/hpylori_project/master-data/separated_patches/test_data_full/images
   Confidence threshold: 0.3
   NMS IoU threshold: 0.2 (20% overlap - aggressive)
   Output directory: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934

🔬 FASTER R-CNN SINGLE WSI TEST - WSI 522934
Device: cuda
GPU: Tesla V100-PCIE-32GB

🔥 Loading Model: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/models/rcnn_bacteria_best_f1.pth
✅ Model loaded from checkpoint
   Epoch: 42
   F1 Score: 0.5285
   Precision: 0.4924
   Recall: 0.5704

🚀 STARTING SINGLE WSI TEST - 522934

🔍 Found 17,670 patches for WSI 522934

⚡ Running R-CNN Inference (Conf > 0.3)...


Running R-CNN: 100%|█████████████████████████| 1105/1105 [09:38<00:00,  1.91it/s]



📊 Initial detections: 1,374

🔍 Analyzing detections before NMS...
   Detections from 1142 unique patches
   Avg detections per patch: 1.2
🔧 Applying NMS (IoU threshold: 0.2)...
   Before NMS: 1,374 boxes
   Score range: 0.300 - 0.961
   🗑️  NMS removed 202 overlapping boxes
   After NMS:  1,172 boxes
   Removed:    202 overlapping boxes (14.7%)
   ✅ NMS successfully removed overlaps

📊 FINAL RESULTS FOR WSI 522934
   Total patches scanned: 17,670
   Total bacteria detected: 1,172

🦠 Detection Statistics:
   Max confidence: 0.961
   Avg confidence: 0.601

📈 Confidence Distribution:
   Very High (>0.7): 428 bacteria
   High (0.5-0.7): 277 bacteria
   Medium (0.3-0.5): 467 bacteria

💾 Results saved: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/522934_detections.json

📄 Generating ImageScope XML...
   ✅ XML saved: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/522934.xml
   📊 Total annotations: 1,172

🎨 Creating visualiza

In [ ]:
Model verification

In [1]:
#!/usr/bin/env python3
"""
Faster R-CNN vs Expert Verification - Single Slide Analysis

Compares Faster R-CNN predictions with expert-verified annotations
for slide 522934 to calculate model performance metrics.

Author: Automated verification script
Date: 2026-02-09
"""

import xml.etree.ElementTree as ET
from pathlib import Path
import json
import shutil
from datetime import datetime
import re

# ====================================================================
# CONFIGURATION
# ====================================================================

# Paths
PREDICTION_XML = Path("/home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/522934.xml")
EXPERT_XML = Path("/home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/522934_RCNN_PO.xml")
PATCHES_DIR = Path("/home/biopsy_gregorova/hpylori_project/master-data/separated_patches/test_data_full/images")

# Output directory (NEW - will not overwrite existing files)
OUTPUT_DIR = Path("/home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/verification_results")

# WSI Information
WSI_ID = "522934"

# Model Parameters (for reference in report)
CONF_THRESHOLD = 0.6
IOU_THRESHOLD = 0.2  # NMS threshold
PATCH_SIZE = 512

# Verification Parameters
MATCH_IOU_THRESHOLD = 0.3  # IoU threshold for matching prediction to ground truth (30%)

# ====================================================================
# XML PARSING
# ====================================================================

def parse_aperio_xml(xml_path):
    """
    Extract rectangular annotations from Aperio XML format.
    
    Args:
        xml_path: Path to XML file
        
    Returns:
        List of annotation dictionaries with bounding box coordinates
    """
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = []
    
    for region in root.findall('.//Region'):
        vertices = []
        for vertex in region.findall('.//Vertex'):
            x = int(float(vertex.get('X')))
            y = int(float(vertex.get('Y')))
            vertices.append((x, y))
        
        if len(vertices) >= 4:
            xs = [v[0] for v in vertices]
            ys = [v[1] for v in vertices]
            
            annotation = {
                'x_min': min(xs),
                'y_min': min(ys),
                'x_max': max(xs),
                'y_max': max(ys),
                'id': region.get('Id', 'unknown'),
                'type': region.get('Type', 'unknown')
            }
            
            # Calculate center and dimensions for reporting
            annotation['center_x'] = (annotation['x_min'] + annotation['x_max']) / 2
            annotation['center_y'] = (annotation['y_min'] + annotation['y_max']) / 2
            annotation['width'] = annotation['x_max'] - annotation['x_min']
            annotation['height'] = annotation['y_max'] - annotation['y_min']
            annotation['area'] = annotation['width'] * annotation['height']
            
            annotations.append(annotation)
    
    return annotations


def parse_filename(filename):
    """
    Extract WSI ID and coordinates from patch filename.
    
    Args:
        filename: Patch filename (e.g., "522934_x1024_y2048.png")
        
    Returns:
        Tuple of (wsi_id, offset_x, offset_y)
    """
    match = re.search(r'(.+?)_x(\d+)_y(\d+)', filename.lower())
    if match:
        return match.group(1), int(match.group(2)), int(match.group(3))
    return filename.replace('.png', ''), 0, 0


# ====================================================================
# GEOMETRY & MATCHING
# ====================================================================

def calculate_iou(box1, box2):
    """
    Calculate Intersection over Union (IoU) between two bounding boxes.
    
    Args:
        box1, box2: Dictionaries with keys 'x_min', 'y_min', 'x_max', 'y_max'
        
    Returns:
        IoU value (0.0 to 1.0)
    """
    # Calculate intersection coordinates
    x_left = max(box1['x_min'], box2['x_min'])
    y_top = max(box1['y_min'], box2['y_min'])
    x_right = min(box1['x_max'], box2['x_max'])
    y_bottom = min(box1['y_max'], box2['y_max'])
    
    # Check if there's no intersection
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    # Calculate intersection area
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    
    # Calculate union area
    box1_area = (box1['x_max'] - box1['x_min']) * (box1['y_max'] - box1['y_min'])
    box2_area = (box2['x_max'] - box2['x_min']) * (box2['y_max'] - box2['y_min'])
    union_area = box1_area + box2_area - intersection_area
    
    # Calculate IoU
    if union_area == 0:
        return 0.0
    
    iou = intersection_area / union_area
    return iou


def match_annotations(predictions, ground_truth, iou_threshold=0.3):
    """
    Match predictions to ground truth annotations using IoU.
    
    Args:
        predictions: List of prediction annotations
        ground_truth: List of expert-verified annotations
        iou_threshold: Minimum IoU to consider a match
        
    Returns:
        Dictionary with matched pairs, unmatched predictions, and unmatched GT
    """
    matched_pairs = []
    unmatched_predictions = []
    unmatched_ground_truth = list(range(len(ground_truth)))  # Track GT indices
    matched_ground_truth = set()
    
    # For each prediction, find best matching ground truth
    for pred_idx, pred in enumerate(predictions):
        best_iou = 0.0
        best_gt_idx = -1
        
        for gt_idx, gt in enumerate(ground_truth):
            if gt_idx in matched_ground_truth:
                continue  # Skip already matched GT
            
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        # Check if we found a match
        if best_iou >= iou_threshold:
            matched_pairs.append({
                'pred_idx': pred_idx,
                'gt_idx': best_gt_idx,
                'iou': best_iou,
                'prediction': pred,
                'ground_truth': ground_truth[best_gt_idx]
            })
            matched_ground_truth.add(best_gt_idx)
        else:
            unmatched_predictions.append({
                'pred_idx': pred_idx,
                'prediction': pred,
                'best_iou': best_iou
            })
    
    # Remaining unmatched ground truth
    unmatched_gt = [
        {
            'gt_idx': idx,
            'ground_truth': ground_truth[idx]
        }
        for idx in range(len(ground_truth))
        if idx not in matched_ground_truth
    ]
    
    return {
        'true_positives': matched_pairs,
        'false_positives': unmatched_predictions,
        'false_negatives': unmatched_gt
    }


# ====================================================================
# PATCH ANALYSIS
# ====================================================================

def get_patch_bounds(offset_x, offset_y, patch_size=512):
    """Get global coordinates of a patch"""
    return {
        'x_min': offset_x,
        'y_min': offset_y,
        'x_max': offset_x + patch_size,
        'y_max': offset_y + patch_size
    }


def annotation_in_patch(annotation, patch_bounds):
    """Check if annotation overlaps with patch"""
    iou = calculate_iou(annotation, patch_bounds)
    return iou > 0


def find_patches_for_wsi(patches_dir, wsi_id):
    """Find all patches belonging to a specific WSI"""
    all_patches = list(patches_dir.glob("*.png"))
    wsi_patches = []
    
    for patch_file in all_patches:
        patch_wsi_id, offset_x, offset_y = parse_filename(patch_file.name)
        if patch_wsi_id == wsi_id:
            wsi_patches.append({
                'filename': patch_file.name,
                'path': patch_file,
                'offset_x': offset_x,
                'offset_y': offset_y,
                'bounds': get_patch_bounds(offset_x, offset_y)
            })
    
    return wsi_patches


# ====================================================================
# METRICS CALCULATION
# ====================================================================

def calculate_metrics(tp_count, fp_count, fn_count):
    """
    Calculate precision, recall, F1-score, and accuracy.
    
    Args:
        tp_count: True Positives count
        fp_count: False Positives count
        fn_count: False Negatives count
        
    Returns:
        Dictionary with all metrics
    """
    # Precision: TP / (TP + FP)
    precision = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0.0
    
    # Recall (Sensitivity): TP / (TP + FN)
    recall = tp_count / (tp_count + fn_count) if (tp_count + fn_count) > 0 else 0.0
    
    # F1-Score: Harmonic mean of precision and recall
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    # Accuracy: TP / (TP + FP + FN)
    total = tp_count + fp_count + fn_count
    accuracy = tp_count / total if total > 0 else 0.0
    
    return {
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score,
        'accuracy': accuracy,
        'true_positives': tp_count,
        'false_positives': fp_count,
        'false_negatives': fn_count,
        'total_detections': tp_count + fp_count,
        'total_ground_truth': tp_count + fn_count
    }


# ====================================================================
# REPORTING
# ====================================================================

def create_detailed_report(matches, metrics, output_dir):
    """
    Create comprehensive verification report with all details.
    
    Args:
        matches: Dictionary with TP/FP/FN matches
        metrics: Dictionary with calculated metrics
        output_dir: Where to save the report
    """
    report_dir = output_dir / 'reports'
    report_dir.mkdir(parents=True, exist_ok=True)
    
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # ===== TEXT REPORT =====
    text_report_path = report_dir / f'verification_report_{timestamp}.txt'
    
    with open(text_report_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("FASTER R-CNN MODEL VERIFICATION REPORT\n")
        f.write("Single Slide Analysis: WSI 522934\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("FILES ANALYZED:\n")
        f.write(f"  • Model Predictions:  {PREDICTION_XML.name}\n")
        f.write(f"  • Expert Verified:    {EXPERT_XML.name}\n\n")
        
        f.write("MODEL PARAMETERS:\n")
        f.write(f"  • Confidence Threshold: {CONF_THRESHOLD}\n")
        f.write(f"  • NMS IoU Threshold:    {IOU_THRESHOLD}\n")
        f.write(f"  • Matching IoU:         {MATCH_IOU_THRESHOLD}\n\n")
        
        f.write("="*80 + "\n")
        f.write("CLASSIFICATION RESULTS\n")
        f.write("="*80 + "\n\n")
        
        tp = metrics['true_positives']
        fp = metrics['false_positives']
        fn = metrics['false_negatives']
        
        f.write(f"✅ TRUE POSITIVES:  {tp:4d} detections\n")
        f.write(f"   → Model detected, Expert confirmed\n")
        f.write(f"   → Correct detections by the model\n\n")
        
        f.write(f"⚠️  FALSE POSITIVES: {fp:4d} detections\n")
        f.write(f"   → Model detected, Expert removed/rejected\n")
        f.write(f"   → Incorrect detections (model hallucinations)\n\n")
        
        f.write(f"❌ FALSE NEGATIVES: {fn:4d} annotations\n")
        f.write(f"   → Model missed, Expert added\n")
        f.write(f"   → Bacteria that model failed to detect\n\n")
        
        f.write("="*80 + "\n")
        f.write("PERFORMANCE METRICS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Precision:  {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)\n")
        f.write(f"  → Of all detections, {metrics['precision']*100:.1f}% were correct\n")
        f.write(f"  → Formula: TP / (TP + FP) = {tp} / ({tp} + {fp})\n\n")
        
        f.write(f"Recall:     {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%)\n")
        f.write(f"  → Model found {metrics['recall']*100:.1f}% of all bacteria\n")
        f.write(f"  → Formula: TP / (TP + FN) = {tp} / ({tp} + {fn})\n\n")
        
        f.write(f"F1-Score:   {metrics['f1_score']:.4f}\n")
        f.write(f"  → Harmonic mean of Precision and Recall\n")
        f.write(f"  → Balanced performance metric\n\n")
        
        f.write(f"Accuracy:   {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)\n")
        f.write(f"  → Overall detection accuracy\n")
        f.write(f"  → Formula: TP / (TP + FP + FN) = {tp} / ({tp} + {fp} + {fn})\n\n")
        
        f.write("="*80 + "\n")
        f.write("DETAILED BREAKDOWN\n")
        f.write("="*80 + "\n\n")
        
        # True Positives Details
        if matches['true_positives']:
            f.write(f"TRUE POSITIVES ({len(matches['true_positives'])} detections):\n")
            f.write("-" * 80 + "\n")
            for i, match in enumerate(matches['true_positives'][:10], 1):  # Show first 10
                f.write(f"\n  Match #{i}:\n")
                f.write(f"    IoU Score: {match['iou']:.4f}\n")
                pred = match['prediction']
                f.write(f"    Prediction: ({pred['x_min']}, {pred['y_min']}) → ({pred['x_max']}, {pred['y_max']})\n")
                gt = match['ground_truth']
                f.write(f"    Expert GT:  ({gt['x_min']}, {gt['y_min']}) → ({gt['x_max']}, {gt['y_max']})\n")
            
            if len(matches['true_positives']) > 10:
                f.write(f"\n  ... and {len(matches['true_positives']) - 10} more\n")
            f.write("\n")
        
        # False Positives Details
        if matches['false_positives']:
            f.write(f"\nFALSE POSITIVES ({len(matches['false_positives'])} detections):\n")
            f.write("-" * 80 + "\n")
            for i, fp_item in enumerate(matches['false_positives'][:10], 1):
                f.write(f"\n  FP #{i}:\n")
                pred = fp_item['prediction']
                f.write(f"    Location: ({pred['x_min']}, {pred['y_min']}) → ({pred['x_max']}, {pred['y_max']})\n")
                f.write(f"    Size: {pred['width']:.0f} × {pred['height']:.0f} px\n")
                f.write(f"    Best IoU with any GT: {fp_item['best_iou']:.4f}\n")
            
            if len(matches['false_positives']) > 10:
                f.write(f"\n  ... and {len(matches['false_positives']) - 10} more\n")
            f.write("\n")
        
        # False Negatives Details
        if matches['false_negatives']:
            f.write(f"\nFALSE NEGATIVES ({len(matches['false_negatives'])} annotations):\n")
            f.write("-" * 80 + "\n")
            for i, fn_item in enumerate(matches['false_negatives'][:10], 1):
                f.write(f"\n  FN #{i}:\n")
                gt = fn_item['ground_truth']
                f.write(f"    Location: ({gt['x_min']}, {gt['y_min']}) → ({gt['x_max']}, {gt['y_max']})\n")
                f.write(f"    Size: {gt['width']:.0f} × {gt['height']:.0f} px\n")
            
            if len(matches['false_negatives']) > 10:
                f.write(f"\n  ... and {len(matches['false_negatives']) - 10} more\n")
            f.write("\n")
        
        f.write("="*80 + "\n")
        f.write("RECOMMENDATIONS\n")
        f.write("="*80 + "\n\n")
        
        if metrics['precision'] < 0.8:
            f.write("⚠️  LOW PRECISION:\n")
            f.write(f"   → Model has {fp} false positives\n")
            f.write(f"   → Consider increasing confidence threshold (currently {CONF_THRESHOLD})\n")
            f.write(f"   → Review false positive patterns for retraining\n\n")
        
        if metrics['recall'] < 0.8:
            f.write("⚠️  LOW RECALL:\n")
            f.write(f"   → Model missed {fn} bacteria\n")
            f.write(f"   → Consider decreasing confidence threshold\n")
            f.write(f"   → Add more diverse training examples\n\n")
        
        if metrics['f1_score'] >= 0.8:
            f.write("✅ GOOD OVERALL PERFORMANCE:\n")
            f.write(f"   → F1-Score of {metrics['f1_score']:.3f} indicates balanced performance\n")
            f.write(f"   → Model is ready for production use\n\n")
        
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"   ✓ Text report: {text_report_path}")
    
    # ===== JSON REPORT =====
    json_report_path = report_dir / f'verification_metrics_{timestamp}.json'
    
    json_data = {
        'metadata': {
            'wsi_id': WSI_ID,
            'timestamp': datetime.now().isoformat(),
            'prediction_xml': str(PREDICTION_XML),
            'expert_xml': str(EXPERT_XML),
            'model_parameters': {
                'confidence_threshold': CONF_THRESHOLD,
                'nms_iou_threshold': IOU_THRESHOLD,
                'matching_iou_threshold': MATCH_IOU_THRESHOLD
            }
        },
        'metrics': metrics,
        'detailed_results': {
            'true_positives': [
                {
                    'iou': match['iou'],
                    'prediction': {
                        'bbox': [match['prediction']['x_min'], match['prediction']['y_min'],
                                match['prediction']['x_max'], match['prediction']['y_max']],
                        'center': [match['prediction']['center_x'], match['prediction']['center_y']],
                        'size': [match['prediction']['width'], match['prediction']['height']]
                    },
                    'ground_truth': {
                        'bbox': [match['ground_truth']['x_min'], match['ground_truth']['y_min'],
                                match['ground_truth']['x_max'], match['ground_truth']['y_max']],
                        'center': [match['ground_truth']['center_x'], match['ground_truth']['center_y']],
                        'size': [match['ground_truth']['width'], match['ground_truth']['height']]
                    }
                }
                for match in matches['true_positives']
            ],
            'false_positives': [
                {
                    'bbox': [fp['prediction']['x_min'], fp['prediction']['y_min'],
                            fp['prediction']['x_max'], fp['prediction']['y_max']],
                    'center': [fp['prediction']['center_x'], fp['prediction']['center_y']],
                    'size': [fp['prediction']['width'], fp['prediction']['height']],
                    'best_iou': fp['best_iou']
                }
                for fp in matches['false_positives']
            ],
            'false_negatives': [
                {
                    'bbox': [fn['ground_truth']['x_min'], fn['ground_truth']['y_min'],
                            fn['ground_truth']['x_max'], fn['ground_truth']['y_max']],
                    'center': [fn['ground_truth']['center_x'], fn['ground_truth']['center_y']],
                    'size': [fn['ground_truth']['width'], fn['ground_truth']['height']]
                }
                for fn in matches['false_negatives']
            ]
        }
    }
    
    with open(json_report_path, 'w') as f:
        json.dump(json_data, f, indent=2)
    
    print(f"   ✓ JSON report: {json_report_path}")
    
    # ===== CSV SUMMARY =====
    csv_report_path = report_dir / f'metrics_summary_{timestamp}.csv'
    
    with open(csv_report_path, 'w') as f:
        f.write("Metric,Value,Percentage\n")
        f.write(f"Precision,{metrics['precision']:.4f},{metrics['precision']*100:.2f}%\n")
        f.write(f"Recall,{metrics['recall']:.4f},{metrics['recall']*100:.2f}%\n")
        f.write(f"F1-Score,{metrics['f1_score']:.4f},{metrics['f1_score']*100:.2f}%\n")
        f.write(f"Accuracy,{metrics['accuracy']:.4f},{metrics['accuracy']*100:.2f}%\n")
        f.write(f"\nCount,Value\n")
        f.write(f"True Positives,{metrics['true_positives']}\n")
        f.write(f"False Positives,{metrics['false_positives']}\n")
        f.write(f"False Negatives,{metrics['false_negatives']}\n")
        f.write(f"Total Predictions,{metrics['total_detections']}\n")
        f.write(f"Total Ground Truth,{metrics['total_ground_truth']}\n")
    
    print(f"   ✓ CSV summary: {csv_report_path}")
    
    return text_report_path, json_report_path, csv_report_path


# ====================================================================
# MAIN VERIFICATION FUNCTION
# ====================================================================

def verify_single_slide():
    """
    Main verification workflow for single slide analysis.
    """
    print("\n" + "="*80)
    print("🔬 FASTER R-CNN VERIFICATION ANALYSIS")
    print(f"   Slide: {WSI_ID}")
    print("="*80)
    
    # Step 1: Validate input files
    print("\n📂 Step 1: Validating input files...")
    
    if not PREDICTION_XML.exists():
        print(f"   ❌ ERROR: Prediction XML not found: {PREDICTION_XML}")
        return None
    print(f"   ✓ Prediction XML: {PREDICTION_XML.name}")
    
    if not EXPERT_XML.exists():
        print(f"   ❌ ERROR: Expert XML not found: {EXPERT_XML}")
        return None
    print(f"   ✓ Expert XML: {EXPERT_XML.name}")
    
    if not PATCHES_DIR.exists():
        print(f"   ⚠️  Warning: Patches directory not found: {PATCHES_DIR}")
        print(f"   Continuing without patch analysis...")
    else:
        print(f"   ✓ Patches directory: {PATCHES_DIR}")
    
    # Step 2: Load annotations
    print("\n📊 Step 2: Loading annotations...")
    
    predictions = parse_aperio_xml(PREDICTION_XML)
    print(f"   ✓ Model predictions: {len(predictions)} detections")
    
    ground_truth = parse_aperio_xml(EXPERT_XML)
    print(f"   ✓ Expert annotations: {len(ground_truth)} bacteria")
    
    if len(predictions) == 0 and len(ground_truth) == 0:
        print("\n   ⚠️  Both files contain no annotations!")
        print("   Cannot calculate metrics.")
        return None
    
    # Step 3: Match predictions to ground truth
    print(f"\n🔍 Step 3: Matching predictions to ground truth (IoU ≥ {MATCH_IOU_THRESHOLD})...")
    
    matches = match_annotations(predictions, ground_truth, MATCH_IOU_THRESHOLD)
    
    tp_count = len(matches['true_positives'])
    fp_count = len(matches['false_positives'])
    fn_count = len(matches['false_negatives'])
    
    print(f"   ✓ True Positives:  {tp_count}")
    print(f"   ✓ False Positives: {fp_count}")
    print(f"   ✓ False Negatives: {fn_count}")
    
    # Step 4: Calculate metrics
    print("\n📈 Step 4: Calculating performance metrics...")
    
    metrics = calculate_metrics(tp_count, fp_count, fn_count)
    
    print(f"   ✓ Precision: {metrics['precision']:.4f} ({metrics['precision']*100:.2f}%)")
    print(f"   ✓ Recall:    {metrics['recall']:.4f} ({metrics['recall']*100:.2f}%)")
    print(f"   ✓ F1-Score:  {metrics['f1_score']:.4f}")
    print(f"   ✓ Accuracy:  {metrics['accuracy']:.4f} ({metrics['accuracy']*100:.2f}%)")
    
    # Step 5: Create output directory
    print(f"\n💾 Step 5: Creating output directory...")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f"   ✓ Output directory: {OUTPUT_DIR}")
    
    # Step 6: Generate reports
    print("\n📝 Step 6: Generating detailed reports...")
    
    create_detailed_report(matches, metrics, OUTPUT_DIR)
    
    # Step 7: Summary
    print("\n" + "="*80)
    print("📊 VERIFICATION SUMMARY")
    print("="*80)
    
    print(f"\n🎯 Overall Performance:")
    print(f"   • Precision:  {metrics['precision']*100:.1f}% - Of {metrics['total_detections']} detections, {tp_count} were correct")
    print(f"   • Recall:     {metrics['recall']*100:.1f}% - Found {tp_count} of {metrics['total_ground_truth']} bacteria")
    print(f"   • F1-Score:   {metrics['f1_score']:.3f} - Balanced performance metric")
    
    print(f"\n📈 Model Behavior:")
    if fp_count > 0:
        print(f"   ⚠️  {fp_count} false alarms (model detected non-bacteria)")
    else:
        print(f"   ✅ No false alarms!")
    
    if fn_count > 0:
        print(f"   ⚠️  {fn_count} missed bacteria (expert found but model didn't)")
    else:
        print(f"   ✅ All bacteria detected!")
    
    print(f"\n💡 Interpretation:")
    if metrics['precision'] >= 0.9:
        print(f"   ✅ Excellent precision - very few false alarms")
    elif metrics['precision'] >= 0.7:
        print(f"   👍 Good precision - acceptable false alarm rate")
    else:
        print(f"   ⚠️  Low precision - many false alarms, consider higher threshold")
    
    if metrics['recall'] >= 0.9:
        print(f"   ✅ Excellent recall - catches most bacteria")
    elif metrics['recall'] >= 0.7:
        print(f"   👍 Good recall - catches most bacteria")
    else:
        print(f"   ⚠️  Low recall - missing many bacteria, consider lower threshold")
    
    print(f"\n📁 Results saved to: {OUTPUT_DIR}/reports/")
    
    print("\n" + "="*80)
    print("✅ VERIFICATION COMPLETE!")
    print("="*80)
    
    return {
        'metrics': metrics,
        'matches': matches,
        'output_dir': OUTPUT_DIR
    }


# ====================================================================
# MAIN EXECUTION
# ====================================================================

if __name__ == "__main__":
    print("\n" + "="*80)
    print("🔬 FASTER R-CNN MODEL VERIFICATION")
    print("   Single Slide Analysis Tool")
    print("="*80)
    
    print("\n💡 This script will:")
    print("   1. Load model predictions and expert annotations")
    print("   2. Match detections using IoU (Intersection over Union)")
    print("   3. Calculate Precision, Recall, F1-Score, and Accuracy")
    print("   4. Generate detailed reports in TXT, JSON, and CSV formats")
    
    print(f"\n📋 Configuration:")
    print(f"   • WSI ID: {WSI_ID}")
    print(f"   • Confidence Threshold: {CONF_THRESHOLD}")
    print(f"   • Matching IoU: {MATCH_IOU_THRESHOLD}")
    print(f"   • Output Directory: {OUTPUT_DIR}")
    
    proceed = input("\n▶️  Start verification? (y/n): ").strip().lower()
    
    if proceed != 'y':
        print("\n❌ Verification cancelled.")
        exit(0)
    
    # Run verification
    results = verify_single_slide()
    
    if results:
        print("\n🎉 Done! Check the reports folder for detailed results.")
        print(f"\n📂 Location: {results['output_dir']}/reports/")
    else:
        print("\n❌ Verification failed. Please check the error messages above.")


🔬 FASTER R-CNN MODEL VERIFICATION
   Single Slide Analysis Tool

💡 This script will:
   1. Load model predictions and expert annotations
   2. Match detections using IoU (Intersection over Union)
   3. Calculate Precision, Recall, F1-Score, and Accuracy
   4. Generate detailed reports in TXT, JSON, and CSV formats

📋 Configuration:
   • WSI ID: 522934
   • Confidence Threshold: 0.6
   • Matching IoU: 0.3
   • Output Directory: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/verification_results



▶️  Start verification? (y/n):  y



🔬 FASTER R-CNN VERIFICATION ANALYSIS
   Slide: 522934

📂 Step 1: Validating input files...
   ✓ Prediction XML: 522934.xml
   ✓ Expert XML: 522934_RCNN_PO.xml
   ✓ Patches directory: /home/biopsy_gregorova/hpylori_project/master-data/separated_patches/test_data_full/images

📊 Step 2: Loading annotations...
   ✓ Model predictions: 1172 detections
   ✓ Expert annotations: 398 bacteria

🔍 Step 3: Matching predictions to ground truth (IoU ≥ 0.3)...
   ✓ True Positives:  377
   ✓ False Positives: 795
   ✓ False Negatives: 21

📈 Step 4: Calculating performance metrics...
   ✓ Precision: 0.3217 (32.17%)
   ✓ Recall:    0.9472 (94.72%)
   ✓ F1-Score:  0.4803
   ✓ Accuracy:  0.3160 (31.60%)

💾 Step 5: Creating output directory...
   ✓ Output directory: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/verification_results

📝 Step 6: Generating detailed reports...
   ✓ Text report: /home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934/veri

In [ ]:
Threshold analysis

In [7]:
#!/usr/bin/env python3
"""
Faster R-CNN Threshold Sensitivity Analysis - Single Slide 522934

Purpose: Analyze how different confidence thresholds affect precision and recall
Similar to YOLO threshold analysis but for R-CNN predictions

Logic:
1. Load R-CNN predictions with confidence scores from JSON
2. Load expert-verified ground truth from XML
3. Test different confidence thresholds (0.1 to 0.95)
4. Calculate Precision & Recall at each threshold
5. Generate graphs showing the trade-off

Output: Precision-Recall curves and metrics CSV
"""

import json
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# ====================================================================
# CONFIGURATION
# ====================================================================

# Input Files
BASE_DIR = Path("/home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934")

# Try to find the JSON file (case-insensitive)
PREDICTIONS_JSON = None
for json_file in BASE_DIR.glob("522934*.json"):
    if 'detection' in json_file.name.lower():
        PREDICTIONS_JSON = json_file
        break

if PREDICTIONS_JSON is None:
    PREDICTIONS_JSON = BASE_DIR / "522934_detections.json"  # Fallback

EXPERT_XML = BASE_DIR / "522934_RCNN_PO.xml"

# Output Directory (NEW - won't overwrite)
OUTPUT_DIR = BASE_DIR / "verification_results/threshold_analysis"

# Analysis Parameters
WSI_ID = "522934"
THRESHOLDS = np.arange(0.1, 0.96, 0.05)  # Test from 0.1 to 0.95 in steps of 0.05
MATCH_IOU_THRESHOLD = 0.3  # IoU threshold for matching prediction to GT

# Model Parameters (from your config)
MODEL_CONF_THRESHOLD = 0.6  # Original threshold used
NMS_IOU_THRESHOLD = 0.2

# ====================================================================
# HELPER FUNCTIONS
# ====================================================================

def calculate_iou(box1, box2):
    """Calculate Intersection over Union between two bounding boxes"""
    x_left = max(box1['x_min'], box2['x_min'])
    y_top = max(box1['y_min'], box2['y_min'])
    x_right = min(box1['x_max'], box2['x_max'])
    y_bottom = min(box1['y_max'], box2['y_max'])
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1['x_max'] - box1['x_min']) * (box1['y_max'] - box1['y_min'])
    box2_area = (box2['x_max'] - box2['x_min']) * (box2['y_max'] - box2['y_min'])
    union_area = box1_area + box2_area - intersection_area
    
    if union_area == 0:
        return 0.0
    
    return intersection_area / union_area


def parse_aperio_xml(xml_path):
    """Extract rectangular annotations from Aperio XML"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = []
    
    for region in root.findall('.//Region'):
        vertices = []
        for vertex in region.findall('.//Vertex'):
            x = int(float(vertex.get('X')))
            y = int(float(vertex.get('Y')))
            vertices.append((x, y))
        
        if len(vertices) >= 4:
            xs = [v[0] for v in vertices]
            ys = [v[1] for v in vertices]
            
            annotations.append({
                'x_min': min(xs),
                'y_min': min(ys),
                'x_max': max(xs),
                'y_max': max(ys),
                'id': region.get('Id', 'unknown')
            })
    
    return annotations


# ====================================================================
# LOAD DATA
# ====================================================================

def load_rcnn_predictions(json_path):
    """
    Load R-CNN predictions from JSON file
    
    Handles multiple JSON structures:
    1. Format with 'detections' list
    2. Format with numbered detections as keys
    3. Flat list format
    
    Returns:
        List of detection dictionaries with bbox and confidence
    """
    print(f"\n📂 Loading R-CNN predictions from JSON...")
    print(f"   Reading: {json_path.name}")
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Extract metadata
    wsi_id = data.get('wsi_id', 'unknown')
    total_patches = data.get('total_patches', 0)
    total_bacteria = data.get('total_bacteria', 0)
    max_conf = data.get('max_confidence', 0)
    avg_conf = data.get('avg_confidence', 0)
    
    print(f"   ✓ WSI ID: {wsi_id}")
    print(f"   ✓ Total patches analyzed: {total_patches:,}")
    print(f"   ✓ Total bacteria detected: {total_bacteria:,}")
    print(f"   ✓ Confidence range: {avg_conf:.3f} (avg) to {max_conf:.3f} (max)")
    
    # Extract detections - handle different formats
    detections = []
    
    # Try Format 1: List under 'detections' key
    if 'detections' in data and isinstance(data['detections'], list):
        raw_detections = data['detections']
        for det in raw_detections:
            # Try 'wsi_box' first (your format), then 'bbox'
            bbox = det.get('wsi_box', det.get('bbox', []))
            score = det.get('score', det.get('confidence', 0.0))
            
            if len(bbox) == 4:
                detections.append({
                    'x_min': float(bbox[0]),
                    'y_min': float(bbox[1]),
                    'x_max': float(bbox[2]),
                    'y_max': float(bbox[3]),
                    'confidence': float(score)
                })
    
    # Try Format 2: Numbered keys (e.g., "0", "1", "2", ...)
    elif 'detections' in data and isinstance(data['detections'], dict):
        raw_detections = data['detections']
        for key, det in raw_detections.items():
            bbox = det.get('wsi_box', det.get('bbox', []))
            score = det.get('score', det.get('confidence', 0.0))
            
            if len(bbox) == 4:
                detections.append({
                    'x_min': float(bbox[0]),
                    'y_min': float(bbox[1]),
                    'x_max': float(bbox[2]),
                    'y_max': float(bbox[3]),
                    'confidence': float(score)
                })
    
    # Try Format 3: Look for numbered keys at root level
    else:
        # Check if root has numbered keys like "0", "1", etc.
        numbered_keys = [k for k in data.keys() if k.isdigit()]
        if numbered_keys:
            print(f"   ℹ️  Detected numbered key format ({len(numbered_keys)} entries)")
            for key in numbered_keys:
                det = data[key]
                # The format from your example: [x_min, y_min, x_max, y_max, score]
                if isinstance(det, list) and len(det) >= 5:
                    detections.append({
                        'x_min': float(det[0]),
                        'y_min': float(det[1]),
                        'x_max': float(det[2]),
                        'y_max': float(det[3]),
                        'confidence': float(det[4])
                    })
                # Alternative: dict format
                elif isinstance(det, dict):
                    bbox = det.get('wsi_box', det.get('bbox', []))
                    score = det.get('score', det.get('confidence', 0.0))
                    if len(bbox) == 4:
                        detections.append({
                            'x_min': float(bbox[0]),
                            'y_min': float(bbox[1]),
                            'x_max': float(bbox[2]),
                            'y_max': float(bbox[3]),
                            'confidence': float(score)
                        })
    
    if len(detections) == 0:
        print(f"\n   ⚠️  WARNING: No detections loaded!")
        print(f"   JSON structure:")
        print(f"   Keys: {list(data.keys())[:10]}")
        if 'detections' in data:
            print(f"   Type of 'detections': {type(data['detections'])}")
    
    print(f"   ✓ Loaded {len(detections):,} detections with confidence scores")
    
    return detections, {
        'wsi_id': wsi_id,
        'total_patches': total_patches,
        'total_bacteria': total_bacteria,
        'max_confidence': max_conf,
        'avg_confidence': avg_conf
    }


# ====================================================================
# THRESHOLD ANALYSIS
# ====================================================================

def evaluate_at_threshold(predictions, ground_truth, threshold, iou_threshold=0.3):
    """
    Evaluate model performance at a specific confidence threshold
    
    Args:
        predictions: List of all predictions with confidence scores
        ground_truth: List of expert-verified annotations
        threshold: Confidence threshold to apply
        iou_threshold: IoU threshold for matching
        
    Returns:
        Dictionary with TP, FP, FN counts and metrics
    """
    # Filter predictions by threshold
    filtered_preds = [p for p in predictions if p['confidence'] >= threshold]
    
    if len(filtered_preds) == 0:
        return {
            'threshold': threshold,
            'predictions_count': 0,
            'true_positives': 0,
            'false_positives': 0,
            'false_negatives': len(ground_truth),
            'precision': 0.0,
            'recall': 0.0,
            'f1_score': 0.0
        }
    
    # Match predictions to ground truth
    matched_gt = set()
    true_positives = 0
    
    for pred in filtered_preds:
        best_iou = 0.0
        best_gt_idx = -1
        
        for gt_idx, gt in enumerate(ground_truth):
            if gt_idx in matched_gt:
                continue
            
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        if best_iou >= iou_threshold:
            true_positives += 1
            matched_gt.add(best_gt_idx)
    
    false_positives = len(filtered_preds) - true_positives
    false_negatives = len(ground_truth) - true_positives
    
    # Calculate metrics
    precision = true_positives / len(filtered_preds) if len(filtered_preds) > 0 else 0.0
    recall = true_positives / len(ground_truth) if len(ground_truth) > 0 else 0.0
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    
    return {
        'threshold': threshold,
        'predictions_count': len(filtered_preds),
        'true_positives': true_positives,
        'false_positives': false_positives,
        'false_negatives': false_negatives,
        'precision': precision,
        'recall': recall,
        'f1_score': f1_score
    }


def run_threshold_analysis(predictions, ground_truth, thresholds):
    """
    Run threshold analysis across multiple threshold values
    
    Returns:
        DataFrame with metrics at each threshold
    """
    print(f"\n🔍 Running threshold sensitivity analysis...")
    print(f"   Testing {len(thresholds)} threshold values from {thresholds[0]:.2f} to {thresholds[-1]:.2f}")
    
    results = []
    
    for threshold in thresholds:
        metrics = evaluate_at_threshold(predictions, ground_truth, threshold, MATCH_IOU_THRESHOLD)
        results.append(metrics)
    
    df = pd.DataFrame(results)
    
    print(f"   ✓ Analysis complete")
    
    return df


# ====================================================================
# VISUALIZATION
# ====================================================================

def create_threshold_graphs(metrics_df, output_dir, metadata):
    """
    Create publication-quality graphs showing threshold impact
    
    Creates:
    1. Precision vs Recall trade-off curve
    2. Individual Precision and Recall curves
    3. F1-Score curve
    """
    print(f"\n📊 Generating visualization graphs...")
    
    output_dir.mkdir(parents=True, exist_ok=True)
    sns.set_style("whitegrid")
    
    # ===== GRAPH 1: Precision-Recall Trade-off (Main Paper Graph) =====
    fig, ax1 = plt.subplots(figsize=(12, 7))
    
    # Precision line (Blue)
    ax1.plot(metrics_df['threshold'], metrics_df['precision'] * 100, 
             marker='o', color='#2E86AB', linewidth=2.5, markersize=6, label='Precision')
    ax1.set_xlabel('Confidence Threshold', fontsize=13, fontweight='bold')
    ax1.set_ylabel('Precision (%)', color='#2E86AB', fontsize=13, fontweight='bold')
    ax1.tick_params(axis='y', labelcolor='#2E86AB')
    ax1.set_ylim(-5, 105)
    ax1.grid(True, alpha=0.3)
    
    # Recall line (Green - on second y-axis)
    ax2 = ax1.twinx()
    ax2.plot(metrics_df['threshold'], metrics_df['recall'] * 100, 
             marker='s', color='#06A77D', linewidth=2.5, markersize=6, 
             linestyle='--', label='Recall')
    ax2.set_ylabel('Recall (%)', color='#06A77D', fontsize=13, fontweight='bold')
    ax2.tick_params(axis='y', labelcolor='#06A77D')
    ax2.set_ylim(-5, 105)
    
    # Add vertical line at original threshold
    ax1.axvline(x=MODEL_CONF_THRESHOLD, color='red', linestyle=':', 
                linewidth=2, alpha=0.7, label=f'Model Threshold ({MODEL_CONF_THRESHOLD})')
    
    # Title
    plt.title(f'Faster R-CNN: Precision-Recall Trade-off vs Confidence Threshold\nSlide {WSI_ID}',
              fontsize=14, fontweight='bold', pad=20)
    
    # Combined legend
    lines1, labels1 = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines1 + lines2, labels1 + labels2, loc='center left', fontsize=11, framealpha=0.9)
    
    plt.tight_layout()
    graph1_path = output_dir / 'precision_recall_tradeoff.png'
    plt.savefig(graph1_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: {graph1_path.name}")
    
    # ===== GRAPH 2: F1-Score Curve =====
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(metrics_df['threshold'], metrics_df['f1_score'], 
            marker='D', color='#A23B72', linewidth=2.5, markersize=6)
    ax.set_xlabel('Confidence Threshold', fontsize=13, fontweight='bold')
    ax.set_ylabel('F1-Score', fontsize=13, fontweight='bold')
    ax.set_ylim(-0.05, 1.05)
    ax.grid(True, alpha=0.3)
    
    # Mark the maximum F1-score
    max_f1_idx = metrics_df['f1_score'].idxmax()
    max_f1_threshold = metrics_df.loc[max_f1_idx, 'threshold']
    max_f1_value = metrics_df.loc[max_f1_idx, 'f1_score']
    
    ax.plot(max_f1_threshold, max_f1_value, 'r*', markersize=20, 
            label=f'Best F1 = {max_f1_value:.3f} @ threshold {max_f1_threshold:.2f}')
    
    # Mark actual threshold
    ax.axvline(x=MODEL_CONF_THRESHOLD, color='blue', linestyle=':', 
               linewidth=2, alpha=0.7, label=f'Model Threshold ({MODEL_CONF_THRESHOLD})')
    
    plt.title(f'F1-Score vs Confidence Threshold\nSlide {WSI_ID}',
              fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, framealpha=0.9)
    plt.tight_layout()
    
    graph2_path = output_dir / 'f1_score_curve.png'
    plt.savefig(graph2_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: {graph2_path.name}")
    
    # ===== GRAPH 3: Detection Count vs Threshold =====
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(metrics_df['threshold'], metrics_df['predictions_count'], 
            marker='o', color='#F18F01', linewidth=2.5, markersize=6)
    ax.set_xlabel('Confidence Threshold', fontsize=13, fontweight='bold')
    ax.set_ylabel('Number of Detections', fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)
    
    # Mark actual threshold
    ax.axvline(x=MODEL_CONF_THRESHOLD, color='red', linestyle=':', 
               linewidth=2, alpha=0.7, label=f'Model Threshold ({MODEL_CONF_THRESHOLD})')
    
    plt.title(f'Detection Count vs Confidence Threshold\nSlide {WSI_ID}',
              fontsize=14, fontweight='bold')
    plt.legend(fontsize=11, framealpha=0.9)
    plt.tight_layout()
    
    graph3_path = output_dir / 'detection_count_curve.png'
    plt.savefig(graph3_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: {graph3_path.name}")
    
    return graph1_path, graph2_path, graph3_path


# ====================================================================
# REPORTING
# ====================================================================

def save_analysis_report(metrics_df, output_dir, metadata):
    """
    Save detailed analysis report in multiple formats
    """
    print(f"\n💾 Saving analysis reports...")
    
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # ===== CSV Report =====
    csv_path = output_dir / f'threshold_metrics_{timestamp}.csv'
    
    # Add percentage columns for readability
    export_df = metrics_df.copy()
    export_df['precision_pct'] = export_df['precision'] * 100
    export_df['recall_pct'] = export_df['recall'] * 100
    export_df['f1_score_pct'] = export_df['f1_score'] * 100
    
    export_df.to_csv(csv_path, index=False, float_format='%.4f')
    print(f"   ✓ CSV: {csv_path.name}")
    
    # ===== Text Report =====
    txt_path = output_dir / f'threshold_analysis_{timestamp}.txt'
    
    with open(txt_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("FASTER R-CNN THRESHOLD SENSITIVITY ANALYSIS\n")
        f.write(f"Slide: {WSI_ID}\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("MODEL CONFIGURATION:\n")
        f.write(f"  • Confidence Threshold: {MODEL_CONF_THRESHOLD}\n")
        f.write(f"  • NMS IoU Threshold: {NMS_IOU_THRESHOLD}\n")
        f.write(f"  • Matching IoU Threshold: {MATCH_IOU_THRESHOLD}\n\n")
        
        f.write("DATASET STATISTICS:\n")
        f.write(f"  • Total Patches Analyzed: {metadata['total_patches']:,}\n")
        f.write(f"  • Total Detections (all conf): {metadata['total_bacteria']:,}\n")
        f.write(f"  • Max Confidence: {metadata['max_confidence']:.4f}\n")
        f.write(f"  • Avg Confidence: {metadata['avg_confidence']:.4f}\n\n")
        
        f.write("="*80 + "\n")
        f.write("THRESHOLD ANALYSIS RESULTS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"{'Threshold':<12}{'Detections':<12}{'TP':<8}{'FP':<8}{'FN':<8}{'Precision':<12}{'Recall':<12}{'F1-Score':<12}\n")
        f.write("-"*80 + "\n")
        
        for _, row in metrics_df.iterrows():
            f.write(f"{row['threshold']:<12.2f}")
            f.write(f"{int(row['predictions_count']):<12d}")
            f.write(f"{int(row['true_positives']):<8d}")
            f.write(f"{int(row['false_positives']):<8d}")
            f.write(f"{int(row['false_negatives']):<8d}")
            f.write(f"{row['precision']*100:<12.2f}")
            f.write(f"{row['recall']*100:<12.2f}")
            f.write(f"{row['f1_score']:<12.4f}\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("KEY FINDINGS\n")
        f.write("="*80 + "\n\n")
        
        # Find optimal threshold
        max_f1_idx = metrics_df['f1_score'].idxmax()
        optimal_row = metrics_df.loc[max_f1_idx]
        
        f.write(f"OPTIMAL THRESHOLD (Best F1-Score):\n")
        f.write(f"  • Threshold: {optimal_row['threshold']:.2f}\n")
        f.write(f"  • F1-Score: {optimal_row['f1_score']:.4f}\n")
        f.write(f"  • Precision: {optimal_row['precision']*100:.2f}%\n")
        f.write(f"  • Recall: {optimal_row['recall']*100:.2f}%\n")
        f.write(f"  • Detections: {int(optimal_row['predictions_count'])}\n\n")
        
        # Current threshold performance - find closest threshold to MODEL_CONF_THRESHOLD
        threshold_diffs = abs(metrics_df['threshold'] - MODEL_CONF_THRESHOLD)
        closest_idx = threshold_diffs.idxmin()
        current_row = metrics_df.loc[closest_idx]
        actual_threshold = current_row['threshold']
        
        f.write(f"CURRENT THRESHOLD (~{MODEL_CONF_THRESHOLD}):\n")
        f.write(f"  • Actual threshold tested: {actual_threshold:.2f}\n")
        f.write(f"  • F1-Score: {current_row['f1_score']:.4f}\n")
        f.write(f"  • Precision: {current_row['precision']*100:.2f}%\n")
        f.write(f"  • Recall: {current_row['recall']*100:.2f}%\n")
        f.write(f"  • Detections: {int(current_row['predictions_count'])}\n\n")
        
        # Comparison
        if optimal_row['threshold'] != MODEL_CONF_THRESHOLD:
            f1_improvement = (optimal_row['f1_score'] - current_row['f1_score']) * 100
            f.write(f"POTENTIAL IMPROVEMENT:\n")
            f.write(f"  • F1-Score could improve by {f1_improvement:.2f} percentage points\n")
            f.write(f"  • Consider adjusting threshold to {optimal_row['threshold']:.2f}\n\n")
        else:
            f.write(f"✅ Current threshold is already optimal!\n\n")
        
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"   ✓ Text report: {txt_path.name}")
    
    return csv_path, txt_path


# ====================================================================
# MAIN FUNCTION
# ====================================================================

def main():
    """
    Main execution function
    """
    print("\n" + "="*80)
    print("🔬 FASTER R-CNN THRESHOLD SENSITIVITY ANALYSIS")
    print(f"   Slide: {WSI_ID}")
    print("="*80)
    
    # Step 1: Validate inputs
    print("\n📂 Step 1: Validating input files...")
    
    if not PREDICTIONS_JSON.exists():
        print(f"   ❌ ERROR: Predictions JSON not found: {PREDICTIONS_JSON}")
        print(f"   Looking for files in: {BASE_DIR}")
        json_files = list(BASE_DIR.glob("*.json")) + list(BASE_DIR.glob("*.JSON"))
        if json_files:
            print(f"   Available JSON files:")
            for jf in json_files:
                print(f"      - {jf.name}")
        return
    print(f"   ✓ Predictions JSON: {PREDICTIONS_JSON.name}")
    
    if not EXPERT_XML.exists():
        print(f"   ❌ ERROR: Expert XML not found: {EXPERT_XML}")
        return
    print(f"   ✓ Expert verified XML: {EXPERT_XML.name}")
    
    # Step 2: Load data
    print("\n📊 Step 2: Loading data...")
    
    predictions, metadata = load_rcnn_predictions(PREDICTIONS_JSON)
    ground_truth = parse_aperio_xml(EXPERT_XML)
    
    print(f"\n   Ground Truth:")
    print(f"   ✓ Expert-verified bacteria: {len(ground_truth)} annotations")
    
    # Safety check
    if len(predictions) == 0:
        print(f"\n   ❌ ERROR: No predictions loaded from JSON!")
        print(f"   Cannot perform threshold analysis without predictions.")
        return None
    
    # Step 3: Run analysis
    metrics_df = run_threshold_analysis(predictions, ground_truth, THRESHOLDS)
    
    # Step 4: Create visualizations
    graphs = create_threshold_graphs(metrics_df, OUTPUT_DIR, metadata)
    
    # Step 5: Save reports
    reports = save_analysis_report(metrics_df, OUTPUT_DIR, metadata)
    
    # Step 6: Summary
    print("\n" + "="*80)
    print("📊 ANALYSIS SUMMARY")
    print("="*80)
    
    # Find key thresholds
    max_f1_idx = metrics_df['f1_score'].idxmax()
    optimal_threshold = metrics_df.loc[max_f1_idx, 'threshold']
    optimal_f1 = metrics_df.loc[max_f1_idx, 'f1_score']
    optimal_precision = metrics_df.loc[max_f1_idx, 'precision']
    optimal_recall = metrics_df.loc[max_f1_idx, 'recall']
    
    print(f"\n🎯 OPTIMAL THRESHOLD: {optimal_threshold:.2f}")
    print(f"   • F1-Score:  {optimal_f1:.4f}")
    print(f"   • Precision: {optimal_precision*100:.2f}%")
    print(f"   • Recall:    {optimal_recall*100:.2f}%")
    
    print(f"\n📍 CURRENT THRESHOLD: {MODEL_CONF_THRESHOLD}")
    
    # Find closest tested threshold
    threshold_diffs = abs(metrics_df['threshold'] - MODEL_CONF_THRESHOLD)
    closest_idx = threshold_diffs.idxmin()
    current_metrics = metrics_df.loc[closest_idx]
    actual_threshold = current_metrics['threshold']
    
    if actual_threshold != MODEL_CONF_THRESHOLD:
        print(f"   (Closest tested: {actual_threshold:.2f})")
    
    print(f"   • F1-Score:  {current_metrics['f1_score']:.4f}")
    print(f"   • Precision: {current_metrics['precision']*100:.2f}%")
    print(f"   • Recall:    {current_metrics['recall']*100:.2f}%")
    
    if optimal_threshold != actual_threshold:
        improvement = (optimal_f1 - current_metrics['f1_score']) * 100
        print(f"\n💡 RECOMMENDATION:")
        print(f"   Changing threshold from {MODEL_CONF_THRESHOLD} → {optimal_threshold:.2f}")
        print(f"   Could improve F1-Score by {improvement:.2f} percentage points")
    else:
        print(f"\n✅ Current threshold ({actual_threshold:.2f}) is already optimal!")
    
    print(f"\n📂 Results saved to: {OUTPUT_DIR}")
    print("   • Graphs: precision_recall_tradeoff.png, f1_score_curve.png")
    print("   • Data: threshold_metrics_*.csv")
    print("   • Report: threshold_analysis_*.txt")
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80 + "\n")


# ====================================================================
# ENTRY POINT
# ====================================================================

if __name__ == "__main__":
    main()


🔬 FASTER R-CNN THRESHOLD SENSITIVITY ANALYSIS
   Slide: 522934

📂 Step 1: Validating input files...
   ✓ Predictions JSON: 522934_detections.json
   ✓ Expert verified XML: 522934_RCNN_PO.xml

📊 Step 2: Loading data...

📂 Loading R-CNN predictions from JSON...
   Reading: 522934_detections.json
   ✓ WSI ID: 522934
   ✓ Total patches analyzed: 17,670
   ✓ Total bacteria detected: 1,172
   ✓ Confidence range: 0.601 (avg) to 0.961 (max)
   ✓ Loaded 1,172 detections with confidence scores

   Ground Truth:
   ✓ Expert-verified bacteria: 398 annotations

🔍 Running threshold sensitivity analysis...
   Testing 18 threshold values from 0.10 to 0.95
   ✓ Analysis complete

📊 Generating visualization graphs...
   ✓ Saved: precision_recall_tradeoff.png
   ✓ Saved: f1_score_curve.png
   ✓ Saved: detection_count_curve.png

💾 Saving analysis reports...
   ✓ CSV: threshold_metrics_20260209_174914.csv
   ✓ Text report: threshold_analysis_20260209_174914.txt

📊 ANALYSIS SUMMARY

🎯 OPTIMAL THRESHOLD: 0.

In [8]:
#!/usr/bin/env python3
"""
Faster R-CNN: Model Confidence vs Expert Decision Analysis

Purpose: Analyze how expert verification decisions relate to model confidence scores
This answers: "Which confidence ranges are reliable vs unreliable?"

Outputs:
1. Confidence vs Expert Agreement graph
2. False Positive analysis by confidence bin
3. Missed detections analysis
4. Comprehensive recommendations

Author: Analysis script for R-CNN verification
Date: 2026-02-09
"""

import json
import xml.etree.ElementTree as ET
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from collections import defaultdict

# ====================================================================
# CONFIGURATION
# ====================================================================

# Input Files
BASE_DIR = Path("/home/biopsy_gregorova/hpylori_project/rcnn_optimal_final/evaluation_wsi_522934")

# Try to find the JSON file (case-insensitive)
PREDICTIONS_JSON = None
for json_file in BASE_DIR.glob("522934*.json"):
    if 'detection' in json_file.name.lower():
        PREDICTIONS_JSON = json_file
        break

if PREDICTIONS_JSON is None:
    PREDICTIONS_JSON = BASE_DIR / "522934_detections.json"

EXPERT_XML = BASE_DIR / "522934_RCNN_PO.xml"

# Output Directory
OUTPUT_DIR = BASE_DIR / "verification_results/confidence_analysis"

# Analysis Parameters
WSI_ID = "522934"
MATCH_IOU_THRESHOLD = 0.3  # IoU threshold for matching
CONFIDENCE_BINS = [0.0, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]  # Bins for grouping confidence scores

# ====================================================================
# HELPER FUNCTIONS
# ====================================================================

def calculate_iou(box1, box2):
    """Calculate Intersection over Union between two bounding boxes"""
    x_left = max(box1['x_min'], box2['x_min'])
    y_top = max(box1['y_min'], box2['y_min'])
    x_right = min(box1['x_max'], box2['x_max'])
    y_bottom = min(box1['y_max'], box2['y_max'])
    
    if x_right < x_left or y_bottom < y_top:
        return 0.0
    
    intersection_area = (x_right - x_left) * (y_bottom - y_top)
    box1_area = (box1['x_max'] - box1['x_min']) * (box1['y_max'] - box1['y_min'])
    box2_area = (box2['x_max'] - box2['x_min']) * (box2['y_max'] - box2['y_min'])
    union_area = box1_area + box2_area - intersection_area
    
    if union_area == 0:
        return 0.0
    
    return intersection_area / union_area


def parse_aperio_xml(xml_path):
    """Extract rectangular annotations from Aperio XML"""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    annotations = []
    
    for region in root.findall('.//Region'):
        vertices = []
        for vertex in region.findall('.//Vertex'):
            x = int(float(vertex.get('X')))
            y = int(float(vertex.get('Y')))
            vertices.append((x, y))
        
        if len(vertices) >= 4:
            xs = [v[0] for v in vertices]
            ys = [v[1] for v in vertices]
            
            annotations.append({
                'x_min': min(xs),
                'y_min': min(ys),
                'x_max': max(xs),
                'y_max': max(ys),
                'id': region.get('Id', 'unknown')
            })
    
    return annotations


def load_rcnn_predictions(json_path):
    """Load R-CNN predictions from JSON file"""
    print(f"\n📂 Loading R-CNN predictions...")
    
    with open(json_path, 'r') as f:
        data = json.load(f)
    
    # Extract metadata
    metadata = {
        'wsi_id': data.get('wsi_id', 'unknown'),
        'total_patches': data.get('total_patches', 0),
        'total_bacteria': data.get('total_bacteria', 0),
        'max_confidence': data.get('max_confidence', 0),
        'avg_confidence': data.get('avg_confidence', 0)
    }
    
    print(f"   ✓ WSI ID: {metadata['wsi_id']}")
    print(f"   ✓ Total predictions: {metadata['total_bacteria']:,}")
    print(f"   ✓ Confidence range: {metadata['avg_confidence']:.3f} (avg) to {metadata['max_confidence']:.3f} (max)")
    
    # Extract detections
    detections = []
    
    if 'detections' in data and isinstance(data['detections'], list):
        raw_detections = data['detections']
        for det in raw_detections:
            bbox = det.get('wsi_box', det.get('bbox', []))
            score = det.get('score', det.get('confidence', 0.0))
            
            if len(bbox) == 4:
                detections.append({
                    'x_min': float(bbox[0]),
                    'y_min': float(bbox[1]),
                    'x_max': float(bbox[2]),
                    'y_max': float(bbox[3]),
                    'confidence': float(score)
                })
    
    print(f"   ✓ Loaded {len(detections):,} detections with confidence scores")
    
    return detections, metadata


# ====================================================================
# MATCHING & CLASSIFICATION
# ====================================================================

def match_predictions_to_expert(predictions, expert_annotations, iou_threshold=0.3):
    """
    Match model predictions to expert annotations
    
    Returns:
        Dictionary with classified predictions:
        - true_positives: Model correct (expert kept)
        - false_positives: Model wrong (expert removed)
        - false_negatives: Model missed (expert added)
    """
    print(f"\n🔍 Matching predictions to expert annotations (IoU ≥ {iou_threshold})...")
    
    matched_gt = set()
    
    classified_predictions = []
    
    # For each prediction, find best matching expert annotation
    for pred_idx, pred in enumerate(predictions):
        best_iou = 0.0
        best_gt_idx = -1
        
        for gt_idx, gt in enumerate(expert_annotations):
            if gt_idx in matched_gt:
                continue
            
            iou = calculate_iou(pred, gt)
            if iou > best_iou:
                best_iou = iou
                best_gt_idx = gt_idx
        
        # Classify this prediction
        if best_iou >= iou_threshold:
            # TRUE POSITIVE: Expert kept this detection
            classified_predictions.append({
                **pred,
                'expert_decision': 'KEPT',
                'classification': 'TP',
                'iou': best_iou,
                'gt_idx': best_gt_idx
            })
            matched_gt.add(best_gt_idx)
        else:
            # FALSE POSITIVE: Expert removed/rejected this detection
            classified_predictions.append({
                **pred,
                'expert_decision': 'REMOVED',
                'classification': 'FP',
                'iou': best_iou,
                'gt_idx': -1
            })
    
    # Find FALSE NEGATIVES: Expert added these (model missed)
    false_negatives = []
    for gt_idx, gt in enumerate(expert_annotations):
        if gt_idx not in matched_gt:
            false_negatives.append({
                **gt,
                'classification': 'FN',
                'gt_idx': gt_idx
            })
    
    tp_count = sum(1 for p in classified_predictions if p['classification'] == 'TP')
    fp_count = sum(1 for p in classified_predictions if p['classification'] == 'FP')
    fn_count = len(false_negatives)
    
    print(f"   ✓ True Positives (Expert kept):    {tp_count:,}")
    print(f"   ✓ False Positives (Expert removed): {fp_count:,}")
    print(f"   ✓ False Negatives (Expert added):   {fn_count:,}")
    
    return {
        'all_predictions': classified_predictions,
        'false_negatives': false_negatives,
        'true_positives': tp_count,
        'false_positives': fp_count,
        'false_negatives_count': fn_count
    }


# ====================================================================
# CONFIDENCE ANALYSIS
# ====================================================================

def analyze_confidence_bins(classified_predictions, bins):
    """
    Analyze expert decisions across different confidence bins
    
    Returns:
        DataFrame with statistics per confidence bin
    """
    print(f"\n📊 Analyzing confidence bins...")
    
    results = []
    
    for i in range(len(bins) - 1):
        bin_min = bins[i]
        bin_max = bins[i + 1]
        bin_label = f"{bin_min:.1f}-{bin_max:.1f}"
        
        # Filter predictions in this bin
        bin_preds = [p for p in classified_predictions 
                     if bin_min <= p['confidence'] < bin_max]
        
        if len(bin_preds) == 0:
            continue
        
        tp_count = sum(1 for p in bin_preds if p['classification'] == 'TP')
        fp_count = sum(1 for p in bin_preds if p['classification'] == 'FP')
        
        total = len(bin_preds)
        expert_kept_pct = (tp_count / total * 100) if total > 0 else 0
        expert_removed_pct = (fp_count / total * 100) if total > 0 else 0
        
        avg_confidence = np.mean([p['confidence'] for p in bin_preds])
        
        results.append({
            'bin': bin_label,
            'bin_min': bin_min,
            'bin_max': bin_max,
            'total_predictions': total,
            'expert_kept': tp_count,
            'expert_removed': fp_count,
            'kept_percentage': expert_kept_pct,
            'removed_percentage': expert_removed_pct,
            'avg_confidence': avg_confidence,
            'reliability': 'High' if expert_kept_pct >= 90 else 'Medium' if expert_kept_pct >= 70 else 'Low'
        })
    
    df = pd.DataFrame(results)
    
    print(f"   ✓ Analyzed {len(df)} confidence bins")
    
    return df


# ====================================================================
# VISUALIZATION
# ====================================================================

def create_visualizations(bin_analysis, classified_predictions, false_negatives, output_dir):
    """
    Create comprehensive visualizations
    """
    print(f"\n📊 Creating visualizations...")
    
    output_dir.mkdir(parents=True, exist_ok=True)
    sns.set_style("whitegrid")
    
    # ===== GRAPH 1: Expert Agreement by Confidence Bin =====
    fig, ax = plt.subplots(figsize=(12, 7))
    
    x_pos = np.arange(len(bin_analysis))
    width = 0.35
    
    # Stacked bars
    ax.bar(x_pos, bin_analysis['kept_percentage'], width, 
           label='Expert Kept (TP)', color='#06A77D', alpha=0.8)
    ax.bar(x_pos, bin_analysis['removed_percentage'], width, 
           bottom=bin_analysis['kept_percentage'],
           label='Expert Removed (FP)', color='#D62828', alpha=0.8)
    
    ax.set_xlabel('Model Confidence Range', fontsize=13, fontweight='bold')
    ax.set_ylabel('Percentage (%)', fontsize=13, fontweight='bold')
    ax.set_title('Expert Verification Decisions by Model Confidence\nWhich Confidence Ranges Are Reliable?',
                 fontsize=14, fontweight='bold', pad=20)
    ax.set_xticks(x_pos)
    ax.set_xticklabels(bin_analysis['bin'], rotation=0)
    ax.set_ylim(0, 105)
    ax.legend(fontsize=11, loc='upper left')
    
    # Add count labels on bars
    for i, (kept, total) in enumerate(zip(bin_analysis['expert_kept'], bin_analysis['total_predictions'])):
        ax.text(i, 50, f'n={total}', ha='center', va='center', 
                fontsize=10, fontweight='bold', color='white')
    
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    graph1_path = output_dir / 'confidence_vs_expert_agreement.png'
    plt.savefig(graph1_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: {graph1_path.name}")
    
    # ===== GRAPH 2: Confidence Distribution (TP vs FP) =====
    fig, ax = plt.subplots(figsize=(12, 6))
    
    tp_confidences = [p['confidence'] for p in classified_predictions if p['classification'] == 'TP']
    fp_confidences = [p['confidence'] for p in classified_predictions if p['classification'] == 'FP']
    
    ax.hist([tp_confidences, fp_confidences], bins=20, label=['Expert Kept (TP)', 'Expert Removed (FP)'],
            color=['#06A77D', '#D62828'], alpha=0.7, edgecolor='black')
    
    ax.set_xlabel('Model Confidence Score', fontsize=13, fontweight='bold')
    ax.set_ylabel('Count', fontsize=13, fontweight='bold')
    ax.set_title('Distribution of Model Confidence: Kept vs Removed by Expert',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    graph2_path = output_dir / 'confidence_distribution_tp_vs_fp.png'
    plt.savefig(graph2_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: {graph2_path.name}")
    
    # ===== GRAPH 3: Precision by Confidence Threshold =====
    fig, ax = plt.subplots(figsize=(12, 6))
    
    thresholds = np.arange(0.5, 1.0, 0.05)
    precisions = []
    counts = []
    
    for thresh in thresholds:
        above_thresh = [p for p in classified_predictions if p['confidence'] >= thresh]
        if len(above_thresh) > 0:
            tp = sum(1 for p in above_thresh if p['classification'] == 'TP')
            precision = tp / len(above_thresh)
            precisions.append(precision * 100)
            counts.append(len(above_thresh))
        else:
            precisions.append(0)
            counts.append(0)
    
    ax.plot(thresholds, precisions, marker='o', linewidth=2.5, markersize=8, color='#2E86AB')
    ax.set_xlabel('Confidence Threshold', fontsize=13, fontweight='bold')
    ax.set_ylabel('Precision (%)', fontsize=13, fontweight='bold')
    ax.set_title('Model Precision at Different Confidence Thresholds\n(Based on Expert Verification)',
                 fontsize=14, fontweight='bold')
    ax.set_ylim(0, 105)
    ax.grid(True, alpha=0.3)
    
    # Add annotations for key points
    for i, (t, p, c) in enumerate(zip(thresholds, precisions, counts)):
        if i % 2 == 0 and c > 0:  # Show every other point
            ax.annotate(f'{p:.1f}%\n(n={c})', 
                       xy=(t, p), xytext=(0, 10),
                       textcoords='offset points', ha='center',
                       fontsize=8, alpha=0.7)
    
    plt.tight_layout()
    graph3_path = output_dir / 'precision_by_threshold.png'
    plt.savefig(graph3_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"   ✓ Saved: {graph3_path.name}")
    
    return graph1_path, graph2_path, graph3_path


# ====================================================================
# REPORTING
# ====================================================================

def create_comprehensive_report(bin_analysis, classified_predictions, false_negatives, 
                               metadata, output_dir):
    """
    Create detailed text report with insights and recommendations
    """
    print(f"\n💾 Creating comprehensive report...")
    
    output_dir.mkdir(parents=True, exist_ok=True)
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    
    # ===== TEXT REPORT =====
    txt_path = output_dir / f'confidence_analysis_report_{timestamp}.txt'
    
    tp_count = sum(1 for p in classified_predictions if p['classification'] == 'TP')
    fp_count = sum(1 for p in classified_predictions if p['classification'] == 'FP')
    fn_count = len(false_negatives)
    
    # Calculate overall metrics
    precision = tp_count / (tp_count + fp_count) if (tp_count + fp_count) > 0 else 0
    recall = tp_count / (tp_count + fn_count) if (tp_count + fn_count) > 0 else 0
    f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
    # Find optimal confidence threshold
    best_f1 = 0
    best_threshold = 0
    for thresh in np.arange(0.5, 1.0, 0.05):
        above = [p for p in classified_predictions if p['confidence'] >= thresh]
        if len(above) > 0:
            tp = sum(1 for p in above if p['classification'] == 'TP')
            fp_local = len(above) - tp
            fn_local = fn_count + (tp_count - tp)
            
            prec = tp / (tp + fp_local) if (tp + fp_local) > 0 else 0
            rec = tp / (tp + fn_local) if (tp + fn_local) > 0 else 0
            f1_local = 2 * (prec * rec) / (prec + rec) if (prec + rec) > 0 else 0
            
            if f1_local > best_f1:
                best_f1 = f1_local
                best_threshold = thresh
    
    with open(txt_path, 'w') as f:
        f.write("="*80 + "\n")
        f.write("MODEL CONFIDENCE vs EXPERT DECISION ANALYSIS\n")
        f.write(f"Slide: {WSI_ID}\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n\n")
        
        f.write("OVERVIEW:\n")
        f.write(f"  • Total Model Predictions: {len(classified_predictions):,}\n")
        f.write(f"  • Expert Verified (Final): {tp_count + fn_count:,}\n")
        f.write(f"  • Model Confidence Range: {metadata['avg_confidence']:.3f} (avg) to {metadata['max_confidence']:.3f} (max)\n\n")
        
        f.write("="*80 + "\n")
        f.write("EXPERT VERIFICATION RESULTS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"✅ EXPERT KEPT (True Positives):     {tp_count:,} ({tp_count/len(classified_predictions)*100:.1f}%)\n")
        f.write(f"   → Model predictions that were CORRECT\n\n")
        
        f.write(f"❌ EXPERT REMOVED (False Positives): {fp_count:,} ({fp_count/len(classified_predictions)*100:.1f}%)\n")
        f.write(f"   → Model predictions that were WRONG (hallucinations)\n\n")
        
        f.write(f"➕ EXPERT ADDED (False Negatives):   {fn_count:,}\n")
        f.write(f"   → Bacteria that model MISSED entirely\n\n")
        
        f.write("OVERALL METRICS:\n")
        f.write(f"  • Precision: {precision:.4f} ({precision*100:.2f}%)\n")
        f.write(f"  • Recall:    {recall:.4f} ({recall*100:.2f}%)\n")
        f.write(f"  • F1-Score:  {f1:.4f}\n\n")
        
        f.write("="*80 + "\n")
        f.write("CONFIDENCE BIN ANALYSIS\n")
        f.write("="*80 + "\n\n")
        
        f.write("Question: Which confidence ranges are reliable?\n\n")
        
        f.write(f"{'Confidence':<15}{'Total':<10}{'Kept':<10}{'Removed':<10}{'% Kept':<12}{'Reliability':<12}\n")
        f.write("-"*80 + "\n")
        
        for _, row in bin_analysis.iterrows():
            f.write(f"{row['bin']:<15}")
            f.write(f"{row['total_predictions']:<10d}")
            f.write(f"{row['expert_kept']:<10d}")
            f.write(f"{row['expert_removed']:<10d}")
            f.write(f"{row['kept_percentage']:<12.1f}")
            f.write(f"{row['reliability']:<12}\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("KEY INSIGHTS\n")
        f.write("="*80 + "\n\n")
        
        # Find most reliable bin
        most_reliable = bin_analysis.loc[bin_analysis['kept_percentage'].idxmax()]
        least_reliable = bin_analysis.loc[bin_analysis['kept_percentage'].idxmin()]
        
        f.write(f"🎯 MOST RELIABLE CONFIDENCE RANGE:\n")
        f.write(f"   {most_reliable['bin']}: {most_reliable['kept_percentage']:.1f}% expert agreement\n")
        f.write(f"   → {most_reliable['expert_kept']} kept, {most_reliable['expert_removed']} removed\n\n")
        
        f.write(f"⚠️  LEAST RELIABLE CONFIDENCE RANGE:\n")
        f.write(f"   {least_reliable['bin']}: {least_reliable['kept_percentage']:.1f}% expert agreement\n")
        f.write(f"   → {least_reliable['expert_kept']} kept, {least_reliable['expert_removed']} removed\n\n")
        
        f.write(f"💡 OPTIMAL THRESHOLD RECOMMENDATION:\n")
        f.write(f"   Threshold: {best_threshold:.2f}\n")
        f.write(f"   Expected F1-Score: {best_f1:.4f}\n")
        f.write(f"   → This threshold maximizes agreement with expert\n\n")
        
        f.write("="*80 + "\n")
        f.write("FALSE POSITIVE ANALYSIS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Total False Positives: {fp_count}\n\n")
        
        # Analyze FP by confidence
        fp_by_conf = bin_analysis[['bin', 'expert_removed', 'removed_percentage']].copy()
        fp_by_conf = fp_by_conf[fp_by_conf['expert_removed'] > 0].sort_values('expert_removed', ascending=False)
        
        f.write("False Positives by Confidence Range:\n")
        for _, row in fp_by_conf.iterrows():
            f.write(f"  {row['bin']:>10}: {row['expert_removed']:>4d} FPs ({row['removed_percentage']:>5.1f}%)\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("FALSE NEGATIVE ANALYSIS\n")
        f.write("="*80 + "\n\n")
        
        f.write(f"Total False Negatives (Expert Added): {fn_count}\n")
        f.write(f"  → These are bacteria the model completely missed\n")
        f.write(f"  → Represents {fn_count/(tp_count+fn_count)*100:.1f}% of all true bacteria\n\n")
        
        if fn_count > 0:
            # Analyze size of missed detections
            fn_sizes = [(fn['x_max'] - fn['x_min']) * (fn['y_max'] - fn['y_min']) 
                       for fn in false_negatives]
            avg_fn_size = np.mean(fn_sizes)
            
            tp_sizes = [(p['x_max'] - p['x_min']) * (p['y_max'] - p['y_min']) 
                       for p in classified_predictions if p['classification'] == 'TP']
            avg_tp_size = np.mean(tp_sizes) if tp_sizes else 0
            
            f.write(f"Characteristics of Missed Bacteria:\n")
            f.write(f"  • Average size: {avg_fn_size:.0f} px²\n")
            f.write(f"  • Detected bacteria avg size: {avg_tp_size:.0f} px²\n")
            
            if avg_fn_size < avg_tp_size * 0.8:
                f.write(f"  • ⚠️  Missed bacteria tend to be SMALLER\n")
            elif avg_fn_size > avg_tp_size * 1.2:
                f.write(f"  • ⚠️  Missed bacteria tend to be LARGER\n")
            else:
                f.write(f"  • ℹ️  Size similar to detected bacteria\n")
        
        f.write("\n" + "="*80 + "\n")
        f.write("RECOMMENDATIONS\n")
        f.write("="*80 + "\n\n")
        
        f.write("1. CONFIDENCE THRESHOLD:\n")
        f.write(f"   Current model uses all detections (various confidences)\n")
        f.write(f"   → Consider using threshold ≥ {best_threshold:.2f} for best precision-recall balance\n\n")
        
        f.write("2. RELIABILITY ZONES:\n")
        high_rel_bins = bin_analysis[bin_analysis['reliability'] == 'High']
        if not high_rel_bins.empty:
            f.write(f"   HIGH reliability: {', '.join(high_rel_bins['bin'].tolist())}\n")
            f.write(f"   → Predictions in these ranges are usually correct\n\n")
        
        low_rel_bins = bin_analysis[bin_analysis['reliability'] == 'Low']
        if not low_rel_bins.empty:
            f.write(f"   LOW reliability: {', '.join(low_rel_bins['bin'].tolist())}\n")
            f.write(f"   → Predictions in these ranges often need expert review\n\n")
        
        f.write("3. MODEL IMPROVEMENT:\n")
        if fn_count > tp_count * 0.3:
            f.write(f"   ⚠️  High false negative rate ({fn_count} missed)\n")
            f.write(f"   → Consider retraining with more diverse examples\n")
            f.write(f"   → Analyze characteristics of missed bacteria\n\n")
        
        if fp_count > tp_count * 0.5:
            f.write(f"   ⚠️  High false positive rate ({fp_count} wrong)\n")
            f.write(f"   → Consider stricter confidence threshold\n")
            f.write(f"   → Review training data for false patterns\n\n")
        
        f.write("="*80 + "\n")
        f.write("END OF REPORT\n")
        f.write("="*80 + "\n")
    
    print(f"   ✓ Text report: {txt_path.name}")
    
    # ===== CSV EXPORT =====
    csv_path = output_dir / f'confidence_bin_analysis_{timestamp}.csv'
    bin_analysis.to_csv(csv_path, index=False, float_format='%.2f')
    print(f"   ✓ CSV export: {csv_path.name}")
    
    # ===== JSON DETAILED DATA =====
    json_path = output_dir / f'detailed_classification_{timestamp}.json'
    
    export_data = {
        'metadata': metadata,
        'summary': {
            'total_predictions': len(classified_predictions),
            'true_positives': tp_count,
            'false_positives': fp_count,
            'false_negatives': fn_count,
            'precision': float(precision),
            'recall': float(recall),
            'f1_score': float(f1),
            'optimal_threshold': float(best_threshold),
            'optimal_f1': float(best_f1)
        },
        'confidence_bins': bin_analysis.to_dict('records'),
        'sample_false_positives': [
            {
                'confidence': p['confidence'],
                'bbox': [p['x_min'], p['y_min'], p['x_max'], p['y_max']]
            }
            for p in classified_predictions[:20] if p['classification'] == 'FP'
        ],
        'sample_false_negatives': [
            {
                'bbox': [fn['x_min'], fn['y_min'], fn['x_max'], fn['y_max']]
            }
            for fn in false_negatives[:20]
        ]
    }
    
    with open(json_path, 'w') as f:
        json.dump(export_data, f, indent=2)
    
    print(f"   ✓ JSON data: {json_path.name}")
    
    return txt_path, csv_path, json_path


# ====================================================================
# MAIN FUNCTION
# ====================================================================

def main():
    """
    Main execution function
    """
    print("\n" + "="*80)
    print("🔬 MODEL CONFIDENCE vs EXPERT DECISION ANALYSIS")
    print(f"   Slide: {WSI_ID}")
    print("="*80)
    
    print("\n💡 This analysis answers:")
    print("   1. Which confidence ranges are reliable vs unreliable?")
    print("   2. At what confidence did expert disagree with model?")
    print("   3. What did the model miss entirely?")
    print("   4. What's the optimal confidence threshold?")
    
    # Step 1: Validate inputs
    print("\n📂 Step 1: Validating input files...")
    
    if not PREDICTIONS_JSON.exists():
        print(f"   ❌ ERROR: Predictions JSON not found: {PREDICTIONS_JSON}")
        return None
    print(f"   ✓ Predictions JSON: {PREDICTIONS_JSON.name}")
    
    if not EXPERT_XML.exists():
        print(f"   ❌ ERROR: Expert XML not found: {EXPERT_XML}")
        return None
    print(f"   ✓ Expert XML: {EXPERT_XML.name}")
    
    # Step 2: Load data
    print("\n📊 Step 2: Loading data...")
    
    predictions, metadata = load_rcnn_predictions(PREDICTIONS_JSON)
    expert_annotations = parse_aperio_xml(EXPERT_XML)
    
    print(f"   ✓ Expert annotations: {len(expert_annotations)} bacteria")
    
    if len(predictions) == 0:
        print(f"\n   ❌ ERROR: No predictions loaded!")
        return None
    
    # Step 3: Match and classify
    results = match_predictions_to_expert(predictions, expert_annotations, MATCH_IOU_THRESHOLD)
    
    # Step 4: Analyze confidence bins
    print("\n📊 Step 4: Analyzing confidence bins...")
    bin_analysis = analyze_confidence_bins(results['all_predictions'], CONFIDENCE_BINS)
    
    # Step 5: Create visualizations
    graphs = create_visualizations(
        bin_analysis, 
        results['all_predictions'], 
        results['false_negatives'],
        OUTPUT_DIR
    )
    
    # Step 6: Create comprehensive report
    reports = create_comprehensive_report(
        bin_analysis,
        results['all_predictions'],
        results['false_negatives'],
        metadata,
        OUTPUT_DIR
    )
    
    # Step 7: Summary
    print("\n" + "="*80)
    print("📊 ANALYSIS SUMMARY")
    print("="*80)
    
    tp = results['true_positives']
    fp = results['false_positives']
    fn = results['false_negatives_count']
    
    print(f"\n🎯 EXPERT DECISIONS ON MODEL PREDICTIONS:")
    print(f"   ✅ Kept (Correct):   {tp:,} ({tp/(tp+fp)*100:.1f}%)")
    print(f"   ❌ Removed (Wrong):  {fp:,} ({fp/(tp+fp)*100:.1f}%)")
    print(f"   ➕ Added (Missed):   {fn:,}")
    
    print(f"\n💡 CONFIDENCE RELIABILITY:")
    for _, row in bin_analysis.iterrows():
        reliability_icon = "🟢" if row['reliability'] == 'High' else "🟡" if row['reliability'] == 'Medium' else "🔴"
        print(f"   {reliability_icon} {row['bin']:>10}: {row['kept_percentage']:>5.1f}% kept ({row['total_predictions']:>4d} predictions)")
    
    print(f"\n📂 Results saved to: {OUTPUT_DIR}")
    print("   • Graphs: confidence_vs_expert_agreement.png, confidence_distribution_tp_vs_fp.png")
    print("   • Report: confidence_analysis_report_*.txt")
    print("   • Data: confidence_bin_analysis_*.csv, detailed_classification_*.json")
    
    print("\n" + "="*80)
    print("✅ ANALYSIS COMPLETE!")
    print("="*80 + "\n")
    
    return {
        'bin_analysis': bin_analysis,
        'results': results,
        'output_dir': OUTPUT_DIR
    }


# ====================================================================
# ENTRY POINT
# ====================================================================

if __name__ == "__main__":
    main()


🔬 MODEL CONFIDENCE vs EXPERT DECISION ANALYSIS
   Slide: 522934

💡 This analysis answers:
   1. Which confidence ranges are reliable vs unreliable?
   2. At what confidence did expert disagree with model?
   3. What did the model miss entirely?
   4. What's the optimal confidence threshold?

📂 Step 1: Validating input files...
   ✓ Predictions JSON: 522934_detections.json
   ✓ Expert XML: 522934_RCNN_PO.xml

📊 Step 2: Loading data...

📂 Loading R-CNN predictions...
   ✓ WSI ID: 522934
   ✓ Total predictions: 1,172
   ✓ Confidence range: 0.601 (avg) to 0.961 (max)
   ✓ Loaded 1,172 detections with confidence scores
   ✓ Expert annotations: 398 bacteria

🔍 Matching predictions to expert annotations (IoU ≥ 0.3)...
   ✓ True Positives (Expert kept):    377
   ✓ False Positives (Expert removed): 795
   ✓ False Negatives (Expert added):   21

📊 Step 4: Analyzing confidence bins...

📊 Analyzing confidence bins...
   ✓ Analyzed 6 confidence bins

📊 Creating visualizations...
   ✓ Saved: confi